# 03. Baseline Modelling

## 1. Notebook Objective and Modelling Framework

The feature-engineering pipeline produced a chronological, pre-match dataset containing team-strength, form, scheduling and season-context variables for Premier League fixtures.

The purpose of this notebook is to establish a set of honest and interpretable modelling benchmarks before introducing more flexible nonlinear models.

The prediction problem is multiclass classification. For fixture $i$, the target is:

$$
Y_i \in \{H,D,A\},
$$

where:

- $H$ represents a home win;
- $D$ represents a draw;
- $A$ represents an away win.

Rather than predicting only the most likely outcome, the models will estimate a complete probability distribution:

$$
\hat{\mathbf{p}}_i
=
\left(
\hat{p}_{i,H},
\hat{p}_{i,D},
\hat{p}_{i,A}
\right),
$$

subject to:

$$
\hat{p}_{i,H}
+
\hat{p}_{i,D}
+
\hat{p}_{i,A}
=
1.
$$

This probability-based framing is essential because the eventual objective is not merely to classify matches correctly. The project aims to compare model probabilities with bookmaker-implied probabilities and identify whether the model contains information beyond the market.

### Chronological evaluation

Football matches are naturally ordered through time. A model used in practice is trained on past fixtures and applied to future fixtures.

The dataset will therefore be split chronologically rather than randomly.

A random split could allow the training sample to contain matches played after fixtures in the validation or test sets. Even where individual features are pre-match safe, this would produce an unrealistic evaluation because the model would indirectly learn from future football environments.

The planned split is:

- training set: the earliest seasons;
- validation set: the second-most-recent completed season;
- test set: the most-recent completed season.

The validation set will be used for model development and decision-making. The validation set will be used for model development and hyperparameter selection. The test set remains a separate holdout evaluation and does not enter model fitting or hyperparameter selection. Its results are reported for comparison but do not influence modelling decisions.

### Baseline hierarchy

Three initial probability benchmarks will be constructed.

#### Uniform-probability baseline

The simplest benchmark assigns equal probability to every outcome:

$$
\hat{p}_H
=
\hat{p}_D
=
\hat{p}_A
=
\frac{1}{3}.
$$

This model contains no football information and provides a minimum reference point.

#### Historical-frequency baseline

A second benchmark predicts outcomes using their frequencies in the training data:

$$
\hat{p}_k
=
\frac{N_k}{N},
\qquad
k \in \{H,D,A\},
$$

where $N_k$ is the number of training fixtures with outcome $k$ and $N$ is the total number of training fixtures.

This captures the general home advantage and the historical frequency of draws and away wins.

#### Multinomial logistic regression

The first feature-based model will be multinomial logistic regression.

For outcome class $k$, the model assigns a linear score:

$$
z_{i,k}
=
\beta_{0,k}
+
\mathbf{x}_i^\top \boldsymbol{\beta}_k,
$$

where $\mathbf{x}_i$ contains the pre-match predictor variables for fixture $i$.

The class probabilities are obtained through the softmax transformation:

$$
\hat{p}_{i,k}
=
\frac{\exp(z_{i,k})}
{\sum_{j \in \{H,D,A\}} \exp(z_{i,j})}.
$$

Multinomial logistic regression provides a valuable baseline because it is:

- probabilistic;
- interpretable;
- computationally efficient;
- suitable for regularisation;
- capable of showing whether the engineered features add value beyond unconditional outcome frequencies.

### Evaluation metrics

The primary metric will be multiclass log loss:

$$
\operatorname{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
y_{i,k}\log(\hat{p}_{i,k}),
$$

where $y_{i,k}=1$ when fixture $i$ has outcome $k$ and $0$ otherwise.

Log loss rewards models that assign high probability to the observed result and heavily penalises confident incorrect predictions.

The multiclass Brier score will also be calculated:

$$
\operatorname{Brier}
=
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
\left(
\hat{p}_{i,k}-y_{i,k}
\right)^2.
$$

Accuracy and the confusion matrix will be reported as secondary diagnostics. They are useful for interpretation but do not fully evaluate probability quality.

The central benchmark question for this notebook is:

> Do the engineered pre-match features allow a simple multinomial model to produce better out-of-sample probabilities than naive outcome-frequency forecasts?

> **Run instruction:** after opening this notebook, select the project virtual environment as the kernel, then use **Restart Kernel and Run All Cells**. The notebook is designed to execute sequentially from top to bottom.

## 2. Load and Validate the Processed Modelling Dataset

The modelling dataset created in `02_feature_engineering.ipynb` will now be loaded from the project’s processed-data directory.

The preferred input is:

`data/processed/premier_league_model_data.parquet`

Parquet is used because it preserves numeric, nullable-integer and date-related data types more reliably than CSV. The CSV export will remain available as a fallback if the Parquet file cannot be loaded.

Before modelling begins, the dataset must be checked to confirm that:

- the file exists in the expected project directory;
- the dataset contains at least one fixture;
- column names are unique;
- exact duplicate rows are absent;
- the identifier columns are available;
- the target column is present;
- the target contains only `H`, `D` and `A`;
- fixture dates can be parsed successfully;
- rows remain chronologically ordered within each season;
- the exported dataset contains no infinite numeric values.

The notebook will locate the repository root dynamically rather than relying on the notebook’s current working directory. This prevents paths such as `notebooks/data/processed/` from being created accidentally when the notebook is executed from inside the `notebooks` directory.

At this stage, no rows will be removed and no predictors will be transformed. The objective is only to verify that the exported feature-engineering output can be treated as the fixed input for the modelling pipeline.

A successful validation will establish the following primary objects:

- `model_data`: the complete processed modelling dataset;
- `season_column`: the season identifier;
- `date_column`: the fixture date;
- `home_team_column`: the home-team identifier;
- `away_team_column`: the away-team identifier;
- `target_column`: the full-time match result.

These objects will be used throughout the remainder of the notebook.

In [1]:
# ============================================================
# 2. Load and Validate the Processed Modelling Dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate the project root
# ------------------------------------------------------------

def find_project_root(start_path):
    """
    Find the nearest parent directory containing the Git repository.
    """
    start_path = Path(start_path).resolve()

    for directory in [start_path, *start_path.parents]:
        if (directory / ".git").exists():
            return directory

    raise FileNotFoundError(
        "Could not locate the project root containing the .git directory."
    )


project_root = find_project_root(Path.cwd())

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

parquet_path = (
    processed_data_directory
    / "premier_league_model_data.parquet"
)

csv_path = (
    processed_data_directory
    / "premier_league_model_data.csv"
)


# ------------------------------------------------------------
# Load the modelling dataset
# ------------------------------------------------------------

if parquet_path.exists():
    model_data = pd.read_parquet(parquet_path)
    loaded_file = parquet_path
    loaded_format = "Parquet"

elif csv_path.exists():
    model_data = pd.read_csv(csv_path)
    loaded_file = csv_path
    loaded_format = "CSV"

else:
    raise FileNotFoundError(
        "Could not find the processed modelling dataset.\n\n"
        f"Checked:\n- {parquet_path}\n- {csv_path}"
    )


# ------------------------------------------------------------
# Identify essential columns
# ------------------------------------------------------------

def find_first_existing_column(
    dataframe,
    candidates,
    label,
    required=True,
):
    """
    Return the first candidate column present in the DataFrame.
    """
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate

    if required:
        raise KeyError(
            f"Could not identify the {label} column. "
            f"Checked: {candidates}"
        )

    return None


season_column = find_first_existing_column(
    model_data,
    ["Season", "season"],
    label="season",
)

date_column = find_first_existing_column(
    model_data,
    ["Date", "date", "MatchDate", "match_date"],
    label="fixture date",
)

home_team_column = find_first_existing_column(
    model_data,
    ["HomeTeam", "home_team", "Home"],
    label="home-team",
)

away_team_column = find_first_existing_column(
    model_data,
    ["AwayTeam", "away_team", "Away"],
    label="away-team",
)

target_column = find_first_existing_column(
    model_data,
    ["FTR", "Result", "result", "FullTimeResult"],
    label="target",
)


# ------------------------------------------------------------
# Basic structural validation
# ------------------------------------------------------------

assert len(model_data) > 0, (
    "The modelling dataset is empty."
)

assert model_data.columns.is_unique, (
    "The modelling dataset contains duplicate column names."
)

exact_duplicate_rows = int(
    model_data.duplicated().sum()
)

assert exact_duplicate_rows == 0, (
    f"{exact_duplicate_rows} exact duplicate rows were detected."
)


# ------------------------------------------------------------
# Parse and validate fixture dates
# ------------------------------------------------------------

model_data[date_column] = pd.to_datetime(
    model_data[date_column],
    errors="coerce",
    dayfirst=True,
)

assert model_data[date_column].notna().all(), (
    "At least one fixture date could not be parsed."
)


# ------------------------------------------------------------
# Validate fixture identifiers
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

assert model_data[identifier_columns].notna().all().all(), (
    "One or more fixture identifier columns contain missing values."
)

assert (
    model_data[home_team_column]
    != model_data[away_team_column]
).all(), "A fixture cannot contain the same home and away team."

duplicate_fixture_count = int(
    model_data.duplicated(
        subset=identifier_columns,
    ).sum()
)

assert duplicate_fixture_count == 0, (
    f"{duplicate_fixture_count} duplicate fixtures were detected."
)


# ------------------------------------------------------------
# Validate the target
# ------------------------------------------------------------

model_data[target_column] = (
    model_data[target_column]
    .astype("string")
    .str.strip()
    .str.upper()
)

valid_target_values = {"H", "D", "A"}

invalid_target_values = sorted(
    set(
        model_data[target_column]
        .dropna()
        .unique()
    )
    - valid_target_values
)

assert not invalid_target_values, (
    "The target contains invalid values: "
    f"{invalid_target_values}"
)

assert model_data[target_column].notna().all(), (
    "The target contains missing values."
)


# ------------------------------------------------------------
# Confirm chronological ordering within seasons
# ------------------------------------------------------------

chronology_check = (
    model_data
    .groupby(
        season_column,
        sort=False,
    )[date_column]
    .apply(lambda dates: dates.is_monotonic_increasing)
)

assert chronology_check.all(), (
    "Fixtures are not chronologically ordered within every season."
)


# ------------------------------------------------------------
# Check numeric columns for infinite values
# ------------------------------------------------------------

numeric_columns = model_data.select_dtypes(
    include=[np.number]
).columns.tolist()

infinite_value_counts = {
    column: int(
        np.isinf(
            pd.to_numeric(
                model_data[column],
                errors="coerce",
            ).astype(float)
        ).sum()
    )
    for column in numeric_columns
}

columns_with_infinite_values = {
    column: count
    for column, count in infinite_value_counts.items()
    if count > 0
}

assert not columns_with_infinite_values, (
    "Infinite values were detected: "
    f"{columns_with_infinite_values}"
)


# ------------------------------------------------------------
# Create compact validation summaries
# ------------------------------------------------------------

target_distribution = (
    model_data[target_column]
    .value_counts()
    .reindex(["H", "D", "A"])
    .rename_axis("Outcome")
    .reset_index(name="Fixtures")
)

target_distribution["Percentage"] = (
    100
    * target_distribution["Fixtures"]
    / len(model_data)
).round(2)

dataset_summary = pd.DataFrame(
    {
        "Metric": [
            "Fixtures",
            "Columns",
            "Seasons",
            "Earliest fixture",
            "Latest fixture",
            "Numeric columns",
            "Columns with missing values",
        ],
        "Value": [
            f"{len(model_data):,}",
            len(model_data.columns),
            model_data[season_column].nunique(),
            model_data[date_column].min().date(),
            model_data[date_column].max().date(),
            len(numeric_columns),
            int(model_data.isna().any().sum()),
        ],
    }
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Processed modelling dataset loaded successfully.")
print(f"Format: {loaded_format}")
print(f"File: {loaded_file}")
print(f"Shape: {model_data.shape}")
print(f"Target column: {target_column}")

display(dataset_summary)
display(target_distribution)
display(model_data.head(10))

Processed modelling dataset loaded successfully.
Format: Parquet
File: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\data\processed\premier_league_model_data.parquet
Shape: (3800, 75)
Target column: FTR


,Metric,Value
0,Fixtures,"3,800"
1,Columns,75
2,Seasons,10
3,Earliest fixture,2015-08-08
4,Latest fixture,2025-05-25
5,Numeric columns,70
6,Columns with missing values,47


,Outcome,Fixtures,Percentage
0,H,1691,44.5
1,D,886,23.32
2,A,1223,32.18


,Season,Date,HomeTeam,AwayTeam,FTR,HomeEloBefore,AwayEloBefore,HomeRollingPoints5,AwayRollingPoints5,HomeRollingGoalsFor5,...,PositionDifference,GoalDifferenceDifference,HomeTop4Before,HomeTop6Before,HomeTopHalfBefore,HomeBottom3Before,AwayTop4Before,AwayTop6Before,AwayTopHalfBefore,AwayBottom3Before
0,2015-16,2015-08-08,Bournemouth,Aston Villa,A,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2015-16,2015-08-08,Chelsea,Swansea,D,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2015-16,2015-08-08,Everton,Watford,D,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2015-16,2015-08-08,Leicester,Sunderland,H,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2015-16,2015-08-08,Man United,Tottenham,H,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,2015-16,2015-08-08,Norwich,Crystal Palace,A,1500.0,1500.0,NaN,NaN,NaN,...,<NA>,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,2015-16,2015-08-09,Arsenal,West Ham,A,1500.0,1500.0,NaN,NaN,NaN,...,7,0,0,0,1,0,0,0,0,0
7,2015-16,2015-08-09,Newcastle,Southampton,D,1500.0,1500.0,NaN,NaN,NaN,...,1,0,0,0,0,0,0,0,0,0
8,2015-16,2015-08-09,Stoke,Liverpool,A,1500.0,1500.0,NaN,NaN,NaN,...,-4,0,0,0,0,0,0,0,1,0
9,2015-16,2015-08-10,West Brom,Man City,A,1500.0,1500.0,NaN,NaN,NaN,...,-1,0,0,0,0,0,0,0,0,0


### Results and Interpretation

The processed modelling dataset loaded successfully from the project-level `data/processed` directory.

The validation confirms that:

- the dataset contains fixtures and modelling variables;
- all column names are unique;
- no exact duplicate rows are present;
- fixture identifiers are complete;
- no fixture contains the same home and away team;
- the target column contains only `H`, `D` and `A`;
- fixture dates were parsed successfully;
- fixtures remain chronologically ordered within each season;
- no numeric column contains infinite values.

The target distribution provides the first indication of class imbalance in Premier League outcomes.

Home wins are generally the most common result, reflecting the historical home advantage. Draws and away wins occur less frequently, meaning that accuracy alone would be an incomplete measure of model quality.

A model could achieve a superficially reasonable accuracy by predicting the most common outcome too often while still producing poor probabilities for draws and away wins. For this reason, later evaluation will prioritise multiclass log loss and Brier score.

The successfully loaded `model_data` DataFrame now represents the fixed input for the baseline-modelling pipeline.

No rows have been removed and no feature transformations have yet been applied.

## 3. Define Identifiers, Target and Predictor Columns

Before constructing the training, validation and test sets, the columns in `model_data` must be separated according to their modelling role.

The dataset contains three distinct groups:

1. fixture identifiers;
2. the prediction target;
3. eligible pre-match predictors.

### Fixture Identifiers

The identifier columns describe each match and are retained for chronological splitting, interpretation and prediction output:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These columns will not be passed directly into the baseline logistic-regression model.

`Season` and `Date` are required to preserve the temporal structure of the dataset. The team-name columns identify each fixture but are excluded from the initial baseline because representing team identity directly would require an additional categorical-encoding strategy.

Team strength is already represented through quantitative pre-match variables such as Elo ratings, rolling form and league-table state.

### Target Variable

The prediction target is the full-time result:

`FTR`

with possible values:

$$
Y_i \in \{H,D,A\}.
$$

The classes represent:

- `H`: home win;
- `D`: draw;
- `A`: away win.

For consistent probability output, the class order used throughout the project will be:

$$
(H,D,A).
$$

Maintaining a fixed class order is essential because each predicted-probability column must always correspond to the same outcome.

### Predictor Variables

The predictor set consists of the remaining eligible numeric columns in the processed modelling dataset.

These variables describe information available before kickoff, including:

- pre-match Elo ratings;
- general rolling form;
- venue-specific rolling form;
- rest and fixture congestion;
- relative home–away differences;
- season progress;
- reconstructed pre-match league-table state;
- league-position category indicators.

The predictor matrix will be denoted by:

$$
X \in \mathbb{R}^{N \times P},
$$

where:

- $N$ is the number of fixtures;
- $P$ is the number of predictor variables.

The target vector will be denoted by:

$$
\mathbf{y}
=
(y_1,\ldots,y_N).
$$

### Leakage Protection

The predictor set must not contain:

- `FTR`;
- full-time or half-time goals;
- match statistics recorded during the fixture;
- final-season information;
- bookmaker probabilities or odds;
- temporary feature-engineering columns.

The processed dataset was designed to exclude these variables, but this notebook will validate the separation again before modelling.

### Missing Values

Some predictors contain intentional missing values.

For example, league position and league-position category indicators are undefined before the first completed fixture batch of each season.

Later preprocessing will impute missing numeric values using statistics learned from the training set only.

No imputation, scaling or other transformation will be fitted before the chronological split. This prevents information from the validation or test periods from influencing the training pipeline.

This section will create the following objects:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`

It will also produce a feature summary confirming the number, data type and missingness of the available predictors.

In [2]:
# ============================================================
# 3. Define Identifiers, Target and Predictor Columns
# ============================================================

# ------------------------------------------------------------
# Define identifier and target columns
# ------------------------------------------------------------

identifier_columns = [
    season_column,
    date_column,
    home_team_column,
    away_team_column,
]

class_order = ["H", "D", "A"]


# ------------------------------------------------------------
# Identify eligible numeric predictor columns
# ------------------------------------------------------------

excluded_columns = set(
    identifier_columns
    + [target_column]
)

feature_columns = [
    column
    for column in model_data.columns
    if (
        column not in excluded_columns
        and pd.api.types.is_numeric_dtype(model_data[column])
    )
]


# ------------------------------------------------------------
# Validate that predictors have been identified
# ------------------------------------------------------------

assert feature_columns, (
    "No numeric predictor columns were identified."
)

assert len(feature_columns) == len(set(feature_columns)), (
    "The predictor list contains duplicate column names."
)

assert target_column not in feature_columns, (
    "The target column has entered the predictor set."
)

assert not set(identifier_columns).intersection(feature_columns), (
    "One or more identifier columns entered the predictor set."
)


# ------------------------------------------------------------
# Check for obvious leakage columns
# ------------------------------------------------------------

blocked_exact_names = {
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
    "HomeGoals",
    "AwayGoals",
    "HomeScore",
    "AwayScore",
    "FullTimeResult",
}

blocked_name_fragments = [
    "bookmaker",
    "market_probability",
    "marketprob",
    "implied_probability",
    "impliedprob",
    "final_position",
    "finalposition",
]

blocked_predictors = [
    column
    for column in feature_columns
    if (
        column in blocked_exact_names
        or any(
            fragment in column.lower()
            for fragment in blocked_name_fragments
        )
    )
]

assert not blocked_predictors, (
    "Potential leakage columns were detected in the predictor set: "
    f"{blocked_predictors}"
)


# ------------------------------------------------------------
# Construct the predictor matrix and target vector
# ------------------------------------------------------------

X = model_data[feature_columns].copy()
y = model_data[target_column].copy()

fixture_metadata = model_data[
    identifier_columns
].copy()


# ------------------------------------------------------------
# Validate shapes and index alignment
# ------------------------------------------------------------

assert len(X) == len(model_data), (
    "The predictor matrix does not contain every fixture."
)

assert len(y) == len(model_data), (
    "The target vector does not contain every fixture."
)

assert X.index.equals(y.index), (
    "The predictor matrix and target vector are not aligned."
)

assert fixture_metadata.index.equals(X.index), (
    "Fixture metadata and predictors are not aligned."
)

assert X.columns.is_unique, (
    "The predictor matrix contains duplicate columns."
)


# ------------------------------------------------------------
# Validate target classes
# ------------------------------------------------------------

observed_classes = set(
    y.dropna().unique()
)

assert observed_classes == set(class_order), (
    "The observed target classes do not match H, D and A. "
    f"Observed classes: {sorted(observed_classes)}"
)


# ------------------------------------------------------------
# Validate predictor values
# ------------------------------------------------------------

non_numeric_predictors = [
    column
    for column in feature_columns
    if not pd.api.types.is_numeric_dtype(X[column])
]

assert not non_numeric_predictors, (
    "Non-numeric predictor columns were detected: "
    f"{non_numeric_predictors}"
)

infinite_predictor_counts = {}

for column in feature_columns:
    numeric_values = pd.to_numeric(
        X[column],
        errors="coerce",
    ).astype(float)

    infinite_predictor_counts[column] = int(
        np.isinf(numeric_values).sum()
    )

predictors_with_infinite_values = {
    column: count
    for column, count in infinite_predictor_counts.items()
    if count > 0
}

assert not predictors_with_infinite_values, (
    "Infinite values were detected in the predictors: "
    f"{predictors_with_infinite_values}"
)


# ------------------------------------------------------------
# Create the feature summary
# ------------------------------------------------------------

feature_summary = pd.DataFrame(
    {
        "Feature": feature_columns,
        "DataType": [
            str(X[column].dtype)
            for column in feature_columns
        ],
        "MissingValues": [
            int(X[column].isna().sum())
            for column in feature_columns
        ],
        "MissingPercentage": [
            round(
                100 * X[column].isna().mean(),
                2,
            )
            for column in feature_columns
        ],
        "UniqueValues": [
            int(X[column].nunique(dropna=True))
            for column in feature_columns
        ],
        "Minimum": [
            X[column].min(skipna=True)
            for column in feature_columns
        ],
        "Maximum": [
            X[column].max(skipna=True)
            for column in feature_columns
        ],
    }
)

features_with_missing_values = (
    feature_summary[
        feature_summary["MissingValues"] > 0
    ]
    .sort_values(
        by=[
            "MissingPercentage",
            "MissingValues",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Identifiers, target and predictors defined successfully.")
print(f"Fixtures: {len(model_data):,}")
print(f"Identifier columns: {len(identifier_columns)}")
print(f"Target column: {target_column}")
print(f"Predictor columns: {len(feature_columns)}")
print(
    "Predictors containing missing values:",
    len(features_with_missing_values),
)
print(f"Class order: {class_order}")

display(feature_summary)

if not features_with_missing_values.empty:
    display(features_with_missing_values)

Identifiers, target and predictors defined successfully.
Fixtures: 3,800
Identifier columns: 4
Target column: FTR
Predictor columns: 70
Predictors containing missing values: 47
Class order: ['H', 'D', 'A']


,Feature,DataType,MissingValues,MissingPercentage,UniqueValues,Minimum,Maximum
0,HomeEloBefore,float64,0,0.00,3775,1321.589147,1852.597673
1,AwayEloBefore,float64,0,0.00,3776,1316.579895,1854.104860
2,HomeRollingPoints5,float64,502,13.21,15,0.000000,15.000000
3,AwayRollingPoints5,float64,498,13.11,15,0.000000,15.000000
4,HomeRollingGoalsFor5,float64,502,13.21,23,0.000000,24.000000
...,...,...,...,...,...,...,...
65,HomeBottom3Before,Int64,24,0.63,2,0.000000,1.000000
66,AwayTop4Before,Int64,24,0.63,2,0.000000,1.000000
67,AwayTop6Before,Int64,24,0.63,2,0.000000,1.000000
68,AwayTopHalfBefore,Int64,24,0.63,2,0.000000,1.000000


,Feature,DataType,MissingValues,MissingPercentage,UniqueValues,Minimum,Maximum
0,VenuePointsFormDifference5,float64,1016,26.74,30,-14.0,15.0
1,VenueGoalsForFormDifference5,float64,1016,26.74,35,-15.0,20.0
2,VenueGoalsAgainstFormDifference5,float64,1016,26.74,33,-16.0,17.0
3,VenueGoalDifferenceFormDifference5,float64,1016,26.74,50,-24.0,28.0
4,VenueWinRateFormDifference5,float64,1016,26.74,19,-1.0,1.0
5,HomeVenueRollingPoints5,float64,1000,26.32,15,0.0,15.0
6,HomeVenueRollingGoalsFor5,float64,1000,26.32,24,0.0,24.0
7,HomeVenueRollingGoalsAgainst5,float64,1000,26.32,23,0.0,22.0
8,HomeVenueRollingGoalDifference5,float64,1000,26.32,39,-21.0,20.0
9,HomeVenueRollingWinRate5,float64,1000,26.32,6,0.0,1.0


### Results and Interpretation

The modelling columns have now been separated into fixture identifiers, the prediction target and eligible numeric predictors.

The identifier columns are:

- `Season`
- `Date`
- `HomeTeam`
- `AwayTeam`

These variables are retained for chronological splitting, fixture tracking and interpretation, but they are not passed directly into the baseline logistic-regression model.

The target variable is:

- `FTR`

with the fixed class order:

$$
(H,D,A).
$$

This ordering will be preserved whenever predicted probabilities are stored or evaluated, ensuring that each probability column always corresponds to the correct match outcome.

The predictor matrix `X` contains the numeric pre-match variables created during feature engineering, while the target vector `y` contains the observed full-time results.

The validation confirms that:

- the target is not included among the predictors;
- no fixture identifier is included among the predictors;
- every predictor is numeric;
- no predictor contains infinite values;
- predictor names are unique;
- `X`, `y` and `fixture_metadata` contain the same fixtures in the same order;
- all three target classes are present;
- no obvious post-match, bookmaker or final-season variables entered the predictor set.

Some predictor columns contain missing values. These are expected for features that require previous match information or a meaningful pre-match league table.

Missing values have not yet been imputed. Imputation must be learned from the training set only after the chronological split, preventing information from the validation or test periods from influencing the preprocessing pipeline.

The objects now available for modelling are:

- `identifier_columns`
- `target_column`
- `feature_columns`
- `X`
- `y`
- `fixture_metadata`

The next stage is to divide the fixtures into chronological training, validation and test periods.

## 4. Chronological Train–Validation–Test Split

The modelling dataset must now be divided into separate training, validation and test periods.

Because football fixtures occur through time, the split must preserve chronology. A random split would allow the model to train on matches played after fixtures contained in the validation or test sets, producing an unrealistic estimate of future performance.

The split will therefore be performed using complete Premier League seasons.

### Split Design

The seasons will first be ordered according to the date of their earliest fixture.

The dataset will then be divided as follows:

- **training set:** all seasons except the two most recent;
- **validation set:** the second-most-recent season;
- **test set:** the most recent season.

With ten seasons of data, this corresponds to:

- eight seasons for training;
- one season for validation;
- one season for testing.

Let the ordered seasons be:

$$
S_1,S_2,\ldots,S_T.
$$

The training data are:

$$
\mathcal{D}_{\text{train}}
=
\bigcup_{t=1}^{T-2}\mathcal{D}_{S_t},
$$

the validation data are:

$$
\mathcal{D}_{\text{validation}}
=
\mathcal{D}_{S_{T-1}},
$$

and the test data are:

$$
\mathcal{D}_{\text{test}}
=
\mathcal{D}_{S_T}.
$$

### Role of Each Dataset

The training set will be used to:

- estimate preprocessing parameters;
- fit baseline models;
- learn model coefficients.

The validation set will be used to:

- compare modelling choices;
- select regularisation settings;
- assess whether a model improves on the naive benchmarks.

The test set must remain untouched during model development. It will provide the final estimate of performance on the most recent unseen season.

### Preprocessing Discipline

All data-dependent preprocessing must be fitted using the training set only.

This includes:

- missing-value imputation;
- feature scaling;
- any later feature selection;
- model fitting.

For example, if the median of feature $j$ is used for imputation, it must be calculated as:

$$
\widetilde{x}_{j,\text{train}}
=
\operatorname{median}
\left(
X_{\text{train},j}
\right),
$$

and then applied unchanged to the validation and test sets.

Calculating preprocessing statistics from the complete dataset would allow information from future seasons to influence the training process.

### Split Validation

The split will be checked to confirm that:

- every fixture belongs to exactly one dataset;
- no season appears in more than one dataset;
- all training fixtures occur before the validation season;
- all validation fixtures occur before the test season;
- the predictor and target indices remain aligned;
- each dataset contains all three outcome classes;
- the number of fixtures is preserved.

This section will create:

- `train_seasons`
- `validation_season`
- `test_season`
- `X_train`
- `X_validation`
- `X_test`
- `y_train`
- `y_validation`
- `y_test`
- corresponding fixture-metadata objects

The resulting split will remain fixed throughout the later modelling notebooks so that every candidate model is evaluated on the same chronological periods.

In [3]:
# ============================================================
# 4. Chronological Train–Validation–Test Split
# ============================================================

# ------------------------------------------------------------
# Order seasons by the date of their earliest fixture
# ------------------------------------------------------------

season_date_summary = (
    model_data
    .groupby(
        season_column,
        as_index=False,
    )
    .agg(
        SeasonStart=(date_column, "min"),
        SeasonEnd=(date_column, "max"),
        Fixtures=(date_column, "size"),
    )
    .sort_values(
        by="SeasonStart",
        kind="mergesort",
    )
    .reset_index(drop=True)
)

ordered_seasons = (
    season_date_summary[season_column]
    .tolist()
)

assert len(ordered_seasons) >= 3, (
    "At least three seasons are required for separate "
    "training, validation and test sets."
)


# ------------------------------------------------------------
# Define the fixed chronological split
# ------------------------------------------------------------

train_seasons = ordered_seasons[:-2]
validation_season = ordered_seasons[-2]
test_season = ordered_seasons[-1]

assert train_seasons, (
    "The training set must contain at least one season."
)

assert validation_season not in train_seasons, (
    "The validation season appears in the training seasons."
)

assert test_season not in train_seasons, (
    "The test season appears in the training seasons."
)

assert validation_season != test_season, (
    "The validation and test seasons must be different."
)


# ------------------------------------------------------------
# Create split masks
# ------------------------------------------------------------

train_mask = model_data[season_column].isin(
    train_seasons
)

validation_mask = (
    model_data[season_column] == validation_season
)

test_mask = (
    model_data[season_column] == test_season
)


# ------------------------------------------------------------
# Confirm that every fixture belongs to exactly one split
# ------------------------------------------------------------

split_membership_count = (
    train_mask.astype(int)
    + validation_mask.astype(int)
    + test_mask.astype(int)
)

assert (split_membership_count == 1).all(), (
    "Every fixture must belong to exactly one split."
)

assert int(train_mask.sum()) > 0, (
    "The training set is empty."
)

assert int(validation_mask.sum()) > 0, (
    "The validation set is empty."
)

assert int(test_mask.sum()) > 0, (
    "The test set is empty."
)

assert (
    int(train_mask.sum())
    + int(validation_mask.sum())
    + int(test_mask.sum())
    == len(model_data)
), "The split does not preserve every fixture."


# ------------------------------------------------------------
# Construct predictor, target and metadata splits
# ------------------------------------------------------------

X_train = X.loc[train_mask].copy()
X_validation = X.loc[validation_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_validation = y.loc[validation_mask].copy()
y_test = y.loc[test_mask].copy()

metadata_train = fixture_metadata.loc[
    train_mask
].copy()

metadata_validation = fixture_metadata.loc[
    validation_mask
].copy()

metadata_test = fixture_metadata.loc[
    test_mask
].copy()


# ------------------------------------------------------------
# Validate index alignment within each split
# ------------------------------------------------------------

for split_name, split_X, split_y, split_metadata in [
    (
        "training",
        X_train,
        y_train,
        metadata_train,
    ),
    (
        "validation",
        X_validation,
        y_validation,
        metadata_validation,
    ),
    (
        "test",
        X_test,
        y_test,
        metadata_test,
    ),
]:
    assert split_X.index.equals(split_y.index), (
        f"The {split_name} predictors and target are not aligned."
    )

    assert split_X.index.equals(split_metadata.index), (
        f"The {split_name} predictors and metadata are not aligned."
    )

    assert list(split_X.columns) == feature_columns, (
        f"The {split_name} predictor columns changed."
    )


# ------------------------------------------------------------
# Validate that seasons do not overlap across splits
# ------------------------------------------------------------

observed_train_seasons = set(
    metadata_train[season_column].unique()
)

observed_validation_seasons = set(
    metadata_validation[season_column].unique()
)

observed_test_seasons = set(
    metadata_test[season_column].unique()
)

assert observed_train_seasons == set(train_seasons), (
    "The observed training seasons do not match train_seasons."
)

assert observed_validation_seasons == {
    validation_season
}, "The validation set contains an unexpected season."

assert observed_test_seasons == {
    test_season
}, "The test set contains an unexpected season."

assert observed_train_seasons.isdisjoint(
    observed_validation_seasons
), "Training and validation seasons overlap."

assert observed_train_seasons.isdisjoint(
    observed_test_seasons
), "Training and test seasons overlap."

assert observed_validation_seasons.isdisjoint(
    observed_test_seasons
), "Validation and test seasons overlap."


# ------------------------------------------------------------
# Validate chronological separation
# ------------------------------------------------------------

training_end_date = metadata_train[
    date_column
].max()

validation_start_date = metadata_validation[
    date_column
].min()

validation_end_date = metadata_validation[
    date_column
].max()

test_start_date = metadata_test[
    date_column
].min()

assert training_end_date < validation_start_date, (
    "The training period does not end before validation begins."
)

assert validation_end_date < test_start_date, (
    "The validation period does not end before testing begins."
)

for split_name, split_metadata in [
    ("training", metadata_train),
    ("validation", metadata_validation),
    ("test", metadata_test),
]:
    assert split_metadata[date_column].is_monotonic_increasing, (
        f"The {split_name} fixtures are not chronologically ordered."
    )


# ------------------------------------------------------------
# Validate outcome classes in every split
# ------------------------------------------------------------

expected_classes = set(class_order)

for split_name, split_y in [
    ("training", y_train),
    ("validation", y_validation),
    ("test", y_test),
]:
    observed_split_classes = set(
        split_y.unique()
    )

    assert observed_split_classes == expected_classes, (
        f"The {split_name} set does not contain all target classes. "
        f"Observed: {sorted(observed_split_classes)}"
    )


# ------------------------------------------------------------
# Add a split label for later auditing
# ------------------------------------------------------------

split_labels = pd.Series(
    index=model_data.index,
    dtype="string",
    name="DatasetSplit",
)

split_labels.loc[train_mask] = "Train"
split_labels.loc[validation_mask] = "Validation"
split_labels.loc[test_mask] = "Test"

assert split_labels.notna().all(), (
    "At least one fixture has no dataset-split label."
)


# ------------------------------------------------------------
# Create split summary
# ------------------------------------------------------------

split_summary = pd.DataFrame(
    {
        "Split": [
            "Train",
            "Validation",
            "Test",
        ],
        "Seasons": [
            len(train_seasons),
            1,
            1,
        ],
        "SeasonRange": [
            (
                f"{train_seasons[0]} to "
                f"{train_seasons[-1]}"
            ),
            str(validation_season),
            str(test_season),
        ],
        "Fixtures": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            round(
                100 * len(X_train) / len(model_data),
                2,
            ),
            round(
                100 * len(X_validation) / len(model_data),
                2,
            ),
            round(
                100 * len(X_test) / len(model_data),
                2,
            ),
        ],
        "StartDate": [
            metadata_train[date_column].min().date(),
            metadata_validation[date_column].min().date(),
            metadata_test[date_column].min().date(),
        ],
        "EndDate": [
            metadata_train[date_column].max().date(),
            metadata_validation[date_column].max().date(),
            metadata_test[date_column].max().date(),
        ],
    }
)


# ------------------------------------------------------------
# Create class-distribution summary by split
# ------------------------------------------------------------

class_distribution_by_split = pd.concat(
    [
        pd.DataFrame(
            {
                "Split": split_name,
                "Outcome": outcome,
                "Fixtures": int(
                    (split_y == outcome).sum()
                ),
                "Percentage": round(
                    100
                    * (split_y == outcome).mean(),
                    2,
                ),
            },
            index=[0],
        )
        for split_name, split_y in [
            ("Train", y_train),
            ("Validation", y_validation),
            ("Test", y_test),
        ]
        for outcome in class_order
    ],
    ignore_index=True,
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Chronological data split completed successfully.")
print(f"Training seasons: {train_seasons}")
print(f"Validation season: {validation_season}")
print(f"Test season: {test_season}")
print(
    "Split sizes:",
    f"train={len(X_train):,},",
    f"validation={len(X_validation):,},",
    f"test={len(X_test):,}",
)

display(season_date_summary)
display(split_summary)
display(class_distribution_by_split)

Chronological data split completed successfully.
Training seasons: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
Validation season: 2023-24
Test season: 2024-25
Split sizes: train=3,040, validation=380, test=380


,Season,SeasonStart,SeasonEnd,Fixtures
0,2015-16,2015-08-08,2016-05-17,380
1,2016-17,2016-08-13,2017-05-21,380
2,2017-18,2017-08-11,2018-05-13,380
3,2018-19,2018-08-10,2019-05-12,380
4,2019-20,2019-08-09,2020-07-26,380
5,2020-21,2020-09-12,2021-05-23,380
6,2021-22,2021-08-13,2022-05-22,380
7,2022-23,2022-08-05,2023-05-28,380
8,2023-24,2023-08-11,2024-05-19,380
9,2024-25,2024-08-16,2025-05-25,380


,Split,Seasons,SeasonRange,Fixtures,Percentage,StartDate,EndDate
0,Train,8,2015-16 to 2022-23,3040,80.0,2015-08-08,2023-05-28
1,Validation,1,2023-24,380,10.0,2023-08-11,2024-05-19
2,Test,1,2024-25,380,10.0,2024-08-16,2025-05-25


,Split,Outcome,Fixtures,Percentage
0,Train,H,1361,44.77
1,Train,D,711,23.39
2,Train,A,968,31.84
3,Validation,H,175,46.05
4,Validation,D,82,21.58
5,Validation,A,123,32.37
6,Test,H,155,40.79
7,Test,D,93,24.47
8,Test,A,132,34.74


### Results and Interpretation

The chronological train–validation–test split has been completed successfully.

The seasons were ordered using the date of each season’s earliest fixture and then divided into three non-overlapping periods:

- the earliest seasons form the training set;
- the second-most-recent season forms the validation set;
- the most recent season forms the test set.

This design reflects the way the model will be used in practice: information from earlier fixtures is used to predict matches played later in time.

The validation confirms that:

- every fixture belongs to exactly one split;
- no season appears in more than one split;
- the training period ends before the validation period begins;
- the validation period ends before the test period begins;
- predictor, target and metadata indices remain aligned;
- every predictor set contains the same feature columns;
- all three outcomes, `H`, `D` and `A`, appear in every split;
- the total number of fixtures has been preserved;
- fixtures remain chronologically ordered within each split.

The training set will be used to fit preprocessing transformations and estimate model parameters.

The validation set will be used to compare modelling decisions and assess whether a candidate model improves on the initial benchmarks.

The test set will remain untouched during model development. Its purpose is to provide a final estimate of performance on the most recent unseen season.

This separation is especially important for probability modelling. Repeatedly evaluating modelling choices on the test season would gradually leak information about that season into the development process, even if the model were never directly trained on its fixtures.

No imputation or scaling has yet been applied. These transformations will be fitted using `X_train` only and then applied unchanged to `X_validation` and `X_test`.

The fixed chronological split created in this section will be reused throughout the remaining modelling notebooks so that every model is compared using the same historical periods.

## 5. Target Distribution and Class Balance

Before constructing the first probability baselines, the distribution of match outcomes must be examined across the training, validation and test sets.

The target variable contains three possible outcomes:

$$
Y_i \in \{H,D,A\},
$$

where:

- `H` represents a home win;
- `D` represents a draw;
- `A` represents an away win.

Football match outcomes are not evenly distributed. Home wins are typically more common than draws or away wins because of the historical home advantage.

For dataset split $s$ and outcome class $k$, the observed class proportion is:

$$
\widehat{\pi}_{s,k}
=
\frac{N_{s,k}}{N_s},
$$

where:

- $N_{s,k}$ is the number of fixtures in split $s$ with outcome $k$;
- $N_s$ is the total number of fixtures in split $s$.

The class proportions will be calculated separately for:

- the training set;
- the validation set;
- the test set.

### Why Class Balance Matters

Class balance affects both modelling and evaluation.

A model that predicts the most common outcome too frequently may achieve a superficially reasonable accuracy while producing poor probability estimates for draws and away wins.

For example, predicting a home win for every fixture could outperform random classification on accuracy, but it would provide no meaningful assessment of uncertainty.

For this reason, accuracy will remain a secondary metric. The primary metrics will evaluate the complete predicted probability distribution.

### Distribution Shift

The outcome frequencies may change between seasons.

The validation and test sets may contain different proportions of home wins, draws and away wins from the training period because of:

- changes in home advantage;
- variation in team strength;
- unusual seasonal conditions;
- promoted and relegated teams;
- changes in playing style or league competitiveness.

These differences represent genuine temporal distribution shift and should not be corrected using information from future seasons.

### Historical-Frequency Baseline

The training-set outcome proportions will later define the historical-frequency baseline.

For outcome $k$:

$$
\widehat{p}_k
=
\frac{N_{\text{train},k}}
{N_{\text{train}}}.
$$

The same fixed training probabilities will then be assigned to every validation and test fixture.

It would be incorrect to calculate separate baseline probabilities using the validation or test outcomes because those results would not be known when the predictions were made.

### Section Objectives

This section will:

- count each outcome in every dataset split;
- calculate outcome percentages;
- compare class distributions across time;
- identify the majority class;
- quantify any change between training, validation and test periods;
- store the training-set class probabilities for later baseline forecasts.

The main objects created will be:

- `training_class_probabilities`
- `class_balance_summary`
- `class_balance_pivot`

These will provide the foundation for the naive probability benchmarks in the next section.

In [4]:
# ============================================================
# 5. Target Distribution and Class Balance
# ============================================================

# ------------------------------------------------------------
# Store the target vectors for each chronological split
# ------------------------------------------------------------

target_splits = {
    "Train": y_train,
    "Validation": y_validation,
    "Test": y_test,
}


# ------------------------------------------------------------
# Calculate outcome counts and proportions by split
# ------------------------------------------------------------

class_balance_records = []

for split_name, split_target in target_splits.items():

    outcome_counts = (
        split_target
        .value_counts()
        .reindex(
            class_order,
            fill_value=0,
        )
    )

    outcome_proportions = (
        outcome_counts / len(split_target)
    )

    for outcome in class_order:
        class_balance_records.append(
            {
                "Split": split_name,
                "Outcome": outcome,
                "Fixtures": int(
                    outcome_counts.loc[outcome]
                ),
                "Proportion": float(
                    outcome_proportions.loc[outcome]
                ),
                "Percentage": round(
                    100
                    * outcome_proportions.loc[outcome],
                    2,
                ),
            }
        )


class_balance_summary = pd.DataFrame(
    class_balance_records
)


# ------------------------------------------------------------
# Validate the class-balance summary
# ------------------------------------------------------------

assert len(class_balance_summary) == (
    len(target_splits) * len(class_order)
), (
    "The class-balance summary does not contain every "
    "split-outcome combination."
)

for split_name, split_target in target_splits.items():

    split_summary = class_balance_summary[
        class_balance_summary["Split"] == split_name
    ]

    assert set(
        split_summary["Outcome"]
    ) == set(class_order), (
        f"The {split_name} summary does not contain all outcomes."
    )

    assert (
        split_summary["Fixtures"].sum()
        == len(split_target)
    ), (
        f"The {split_name} outcome counts do not match "
        "the number of fixtures."
    )

    assert np.isclose(
        split_summary["Proportion"].sum(),
        1.0,
    ), (
        f"The {split_name} class proportions do not sum to 1."
    )


# ------------------------------------------------------------
# Store training-set probabilities for the later baseline
# ------------------------------------------------------------

training_class_probabilities = (
    y_train
    .value_counts(normalize=True)
    .reindex(class_order)
    .astype(float)
)

assert training_class_probabilities.notna().all(), (
    "At least one target class is missing from the "
    "training probabilities."
)

assert np.isclose(
    training_class_probabilities.sum(),
    1.0,
), "Training class probabilities do not sum to 1."

assert (
    training_class_probabilities >= 0
).all(), "Training class probabilities cannot be negative."

assert (
    training_class_probabilities <= 1
).all(), "Training class probabilities cannot exceed 1."


# ------------------------------------------------------------
# Identify the majority outcome in each split
# ------------------------------------------------------------

majority_class_records = []

for split_name, split_target in target_splits.items():

    split_probabilities = (
        split_target
        .value_counts(normalize=True)
        .reindex(class_order)
    )

    majority_outcome = (
        split_probabilities.idxmax()
    )

    majority_class_records.append(
        {
            "Split": split_name,
            "MajorityOutcome": majority_outcome,
            "MajorityProportion": float(
                split_probabilities.loc[
                    majority_outcome
                ]
            ),
            "MajorityPercentage": round(
                100
                * split_probabilities.loc[
                    majority_outcome
                ],
                2,
            ),
        }
    )


majority_class_summary = pd.DataFrame(
    majority_class_records
)


# ------------------------------------------------------------
# Create a comparison table of class percentages
# ------------------------------------------------------------

class_balance_pivot = (
    class_balance_summary
    .pivot(
        index="Outcome",
        columns="Split",
        values="Percentage",
    )
    .reindex(
        index=class_order,
        columns=[
            "Train",
            "Validation",
            "Test",
        ],
    )
    .reset_index()
)


# ------------------------------------------------------------
# Quantify distribution shift relative to training
# ------------------------------------------------------------

class_balance_shift = (
    class_balance_pivot
    .copy()
)

class_balance_shift[
    "ValidationMinusTrain"
] = (
    class_balance_shift["Validation"]
    - class_balance_shift["Train"]
)

class_balance_shift[
    "TestMinusTrain"
] = (
    class_balance_shift["Test"]
    - class_balance_shift["Train"]
)

class_balance_shift[
    "ValidationMinusTrain"
] = class_balance_shift[
    "ValidationMinusTrain"
].round(2)

class_balance_shift[
    "TestMinusTrain"
] = class_balance_shift[
    "TestMinusTrain"
].round(2)


# ------------------------------------------------------------
# Create a compact training-probability table
# ------------------------------------------------------------

training_probability_table = pd.DataFrame(
    {
        "Outcome": class_order,
        "Probability": [
            training_class_probabilities.loc[outcome]
            for outcome in class_order
        ],
        "Percentage": [
            round(
                100
                * training_class_probabilities.loc[outcome],
                2,
            )
            for outcome in class_order
        ],
    }
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert (
    training_probability_table["Probability"] >= 0
).all(), "A baseline probability is negative."

assert (
    training_probability_table["Probability"] <= 1
).all(), "A baseline probability exceeds 1."

assert np.isclose(
    training_probability_table[
        "Probability"
    ].sum(),
    1.0,
), "The baseline probabilities do not sum to 1."


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Target distribution analysis completed successfully.")
print(
    "Training-set majority outcome:",
    majority_class_summary.loc[
        majority_class_summary["Split"] == "Train",
        "MajorityOutcome",
    ].iloc[0],
)
print(
    "Training class probabilities:",
    {
        outcome: round(
            training_class_probabilities.loc[outcome],
            4,
        )
        for outcome in class_order
    },
)

display(class_balance_summary)

display(class_balance_pivot)

display(class_balance_shift)

display(majority_class_summary)

display(training_probability_table)

Target distribution analysis completed successfully.
Training-set majority outcome: H
Training class probabilities: {'H': np.float64(0.4477), 'D': np.float64(0.2339), 'A': np.float64(0.3184)}


,Split,Outcome,Fixtures,Proportion,Percentage
0,Train,H,1361,0.447697,44.77
1,Train,D,711,0.233882,23.39
2,Train,A,968,0.318421,31.84
3,Validation,H,175,0.460526,46.05
4,Validation,D,82,0.215789,21.58
5,Validation,A,123,0.323684,32.37
6,Test,H,155,0.407895,40.79
7,Test,D,93,0.244737,24.47
8,Test,A,132,0.347368,34.74


Split,Outcome,Train,Validation,Test
0,H,44.77,46.05,40.79
1,D,23.39,21.58,24.47
2,A,31.84,32.37,34.74


Split,Outcome,Train,Validation,Test,ValidationMinusTrain,TestMinusTrain
0,H,44.77,46.05,40.79,1.28,-3.98
1,D,23.39,21.58,24.47,-1.81,1.08
2,A,31.84,32.37,34.74,0.53,2.90


,Split,MajorityOutcome,MajorityProportion,MajorityPercentage
0,Train,H,0.447697,44.77
1,Validation,H,0.460526,46.05
2,Test,H,0.407895,40.79


,Outcome,Probability,Percentage
0,H,0.447697,44.77
1,D,0.233882,23.39
2,A,0.318421,31.84


### Results and Interpretation

The target-distribution analysis confirms that Premier League match outcomes are not evenly balanced across the three classes.

The training, validation and test sets each contain:

- home wins, represented by `H`;
- draws, represented by `D`;
- away wins, represented by `A`.

The `class_balance_summary` table reports the number and proportion of each outcome within every chronological split, while `class_balance_pivot` allows the percentages to be compared directly across time.

Home wins form the majority class in the training period. This reflects the historical home advantage present in Premier League football.

However, the class proportions are not identical across the training, validation and test seasons. The `class_balance_shift` table measures these changes in percentage points relative to the training set:

$$
\Delta_{s,k}
=
\widehat{\pi}_{s,k}
-
\widehat{\pi}_{\text{train},k},
$$

where $s$ represents either the validation or test set and $k$ represents one of the three outcomes.

These differences indicate genuine temporal distribution shift. The balance between home wins, draws and away wins varies from season to season because of changing teams, league competitiveness and broader football conditions.

The future models should not attempt to remove this shift using information from the validation or test outcomes. Instead, they must learn from the historical training period and be evaluated on how well they generalise to the later seasons.

The training-set class proportions have been stored in:

`training_class_probabilities`

These probabilities will form the historical-frequency benchmark in the next section.

For every future fixture, that benchmark will assign the same probability vector:

$$
\widehat{\mathbf{p}}_{\text{frequency}}
=
\left(
\widehat{p}_{H,\text{train}},
\widehat{p}_{D,\text{train}},
\widehat{p}_{A,\text{train}}
\right).
$$

Only the training outcomes are used to estimate these probabilities. The validation and test labels remain excluded from the forecasting process.

The majority-class percentages also demonstrate why accuracy alone is insufficient. A model that predicts the most common result for every fixture may achieve non-trivial accuracy, but it would fail to distinguish between matches and would provide poor probability estimates for less common outcomes.

The next section will construct and evaluate naive probability benchmarks using:

- uniform probabilities;
- training-set historical frequencies;
- majority-class predictions.

These benchmarks establish the minimum performance that the feature-based multinomial logistic-regression model must improve upon.

## 6. Naive Probability Benchmarks

Before fitting a feature-based model, several deliberately simple benchmarks must be evaluated.

These benchmarks provide reference levels for probability quality. A more complex model is only useful if it produces better out-of-sample forecasts than methods requiring little or no football information.

The benchmarks will be evaluated on both the validation and test sets using the same fixed chronological split established earlier.

### Uniform-Probability Baseline

The uniform baseline assigns equal probability to every possible outcome:

$$
\widehat{p}_H
=
\widehat{p}_D
=
\widehat{p}_A
=
\frac{1}{3}.
$$

Every fixture therefore receives the probability vector:

$$
\widehat{\mathbf{p}}_{\text{uniform}}
=
\left(
\frac{1}{3},
\frac{1}{3},
\frac{1}{3}
\right).
$$

This benchmark contains no information about football, home advantage, team strength or historical outcome frequencies.

Its purpose is to establish the performance of a completely uninformative probability forecast.

### Historical-Frequency Baseline

The historical-frequency baseline assigns probabilities using the outcome proportions observed in the training set.

For outcome class $k$:

$$
\widehat{p}_{k,\text{train}}
=
\frac{N_{k,\text{train}}}
{N_{\text{train}}},
$$

where:

- $N_{k,\text{train}}$ is the number of training fixtures with outcome $k$;
- $N_{\text{train}}$ is the total number of training fixtures.

Every validation and test fixture receives the same probability vector:

$$
\widehat{\mathbf{p}}_{\text{frequency}}
=
\left(
\widehat{p}_{H,\text{train}},
\widehat{p}_{D,\text{train}},
\widehat{p}_{A,\text{train}}
\right).
$$

This benchmark captures the unconditional historical home advantage and the long-run frequencies of draws and away wins.

It does not distinguish between individual fixtures.

Only training-set outcomes are used to estimate these probabilities. Recalculating them from the validation or test sets would use information that would not have been available when the predictions were made.

### Majority-Class Baseline

The majority-class benchmark predicts the most common training outcome as the most likely result for every fixture.

Let:

$$
k^*
=
\arg\max_{k \in \{H,D,A\}}
\widehat{p}_{k,\text{train}}.
$$

The predicted class is then:

$$
\widehat{Y}_i = k^*
$$

for every fixture $i$.

This benchmark is mainly useful for interpreting accuracy.

A majority-class classifier may achieve a non-trivial percentage of correct predictions despite making no distinction between fixtures. It is therefore not a meaningful probability model by itself.

For probability-based evaluation, the historical-frequency probabilities will be used rather than assigning probability one to the majority class and zero to the others. A deterministic probability vector would produce extremely large or undefined log loss whenever another outcome occurred.

### Evaluation Metrics

The uniform and historical-frequency probability forecasts will be evaluated using multiclass log loss:

$$
\operatorname{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
y_{i,k}
\log
\left(
\widehat{p}_{i,k}
\right).
$$

Lower values indicate better probability forecasts.

The multiclass Brier score will also be calculated:

$$
\operatorname{Brier}
=
\frac{1}{N}
\sum_{i=1}^{N}
\sum_{k \in \{H,D,A\}}
\left(
\widehat{p}_{i,k}
-
y_{i,k}
\right)^2.
$$

Lower Brier scores indicate that the predicted probability vector lies closer to the observed one-hot outcome vector.

Accuracy will be reported as a secondary metric:

$$
\operatorname{Accuracy}
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbb{1}
\left(
\widehat{Y}_i = Y_i
\right).
$$

### Benchmark Requirements

The implementation will confirm that:

- each probability is between zero and one;
- every probability row sums to one;
- probability columns follow the fixed order `(H, D, A)`;
- historical probabilities are estimated from `y_train` only;
- validation and test predictions contain the correct number of fixtures;
- no validation or test outcomes are used to construct the forecasts;
- all metrics are calculated consistently across benchmarks and splits.

This section will create:

- uniform validation and test probabilities;
- historical-frequency validation and test probabilities;
- majority-class validation and test predictions;
- a reusable multiclass Brier-score function;
- a benchmark-results table.

These results will establish the minimum standard that multinomial logistic regression must exceed.

In [5]:
# ============================================================
# 6. Naive Probability Benchmarks
# ============================================================

from sklearn.metrics import accuracy_score, log_loss


# ------------------------------------------------------------
# Reusable multiclass Brier-score function
# ------------------------------------------------------------

def multiclass_brier_score(
    y_true,
    probability_matrix,
    labels,
):
    """
    Calculate the multiclass Brier score.

    The score is the mean squared distance between each predicted
    probability vector and the observed one-hot outcome vector.
    """
    y_true = pd.Series(y_true).reset_index(drop=True)

    probabilities = np.asarray(
        probability_matrix,
        dtype=float,
    )

    assert probabilities.shape == (
        len(y_true),
        len(labels),
    ), (
        "The probability matrix has an unexpected shape. "
        f"Expected {(len(y_true), len(labels))}, "
        f"received {probabilities.shape}."
    )

    label_to_position = {
        label: position
        for position, label in enumerate(labels)
    }

    invalid_labels = sorted(
        set(y_true.unique()) - set(labels)
    )

    assert not invalid_labels, (
        "The target contains labels that are not included in "
        f"the supplied class order: {invalid_labels}"
    )

    observed_one_hot = np.zeros_like(
        probabilities,
        dtype=float,
    )

    observed_positions = (
        y_true
        .map(label_to_position)
        .to_numpy()
    )

    observed_one_hot[
        np.arange(len(y_true)),
        observed_positions,
    ] = 1.0

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - observed_one_hot
                ) ** 2,
                axis=1,
            )
        )
    )


# ------------------------------------------------------------
# Reusable class-order-safe multiclass log-loss function
# ------------------------------------------------------------

def multiclass_log_loss(
    y_true,
    probability_frame,
    labels,
):
    """
    Calculate multiclass log loss while explicitly matching each
    observed outcome to its corresponding probability column.

    The project's display order is (H, D, A), whereas scikit-learn
    internally assumes lexicographic class ordering when interpreting
    probability-matrix columns. This helper avoids ambiguity by
    calculating the row-level loss directly, then independently checks
    the result against scikit-learn using explicitly sorted columns.
    """
    y_true = pd.Series(y_true).copy()

    assert isinstance(
        probability_frame,
        pd.DataFrame,
    ), "Log-loss probabilities must be stored in a pandas DataFrame."

    assert probability_frame.index.equals(
        y_true.index
    ), "The probability rows and target rows are not aligned."

    assert list(probability_frame.columns) == list(labels), (
        "The probability columns are not in the required project "
        f"order: {list(labels)}."
    )

    invalid_labels = sorted(
        set(y_true.dropna().unique()) - set(labels)
    )

    assert not invalid_labels, (
        "The target contains labels that are absent from the "
        f"probability columns: {invalid_labels}"
    )

    probability_values = probability_frame.to_numpy(dtype=float)

    assert np.isfinite(probability_values).all(), (
        "The probability frame contains non-finite values."
    )

    assert (probability_values >= 0).all(), (
        "The probability frame contains a value below zero."
    )

    assert (probability_values <= 1).all(), (
        "The probability frame contains a value above one."
    )

    assert np.allclose(
        probability_values.sum(axis=1),
        1.0,
    ), "The probability rows do not sum to one."

    column_positions = {
        label: position
        for position, label in enumerate(probability_frame.columns)
    }

    observed_positions = np.array(
        [column_positions[outcome] for outcome in y_true],
        dtype=int,
    )

    observed_probabilities = probability_values[
        np.arange(len(y_true)),
        observed_positions,
    ]

    observed_probabilities = np.clip(
        observed_probabilities,
        1e-15,
        1.0,
    )

    manual_log_loss = float(
        -np.mean(np.log(observed_probabilities))
    )

    sklearn_class_order = sorted(labels)

    sklearn_log_loss = log_loss(
        y_true,
        probability_frame[
            sklearn_class_order
        ].to_numpy(dtype=float),
        labels=sklearn_class_order,
    )

    assert np.isclose(
        manual_log_loss,
        sklearn_log_loss,
    ), (
        "The direct and scikit-learn log-loss calculations do not "
        "agree. Check the class order and row alignment."
    )

    return manual_log_loss


# ------------------------------------------------------------
# Helper function: validate probability forecasts
# ------------------------------------------------------------

def validate_probability_forecast(
    probability_frame,
    expected_index,
    labels,
    name,
):
    """
    Confirm that a probability forecast is correctly structured.
    """
    assert isinstance(
        probability_frame,
        pd.DataFrame,
    ), f"{name} must be stored as a pandas DataFrame."

    assert probability_frame.index.equals(
        expected_index
    ), f"{name} is not aligned with the target index."

    assert list(
        probability_frame.columns
    ) == list(labels), (
        f"{name} probability columns are not ordered as "
        f"{list(labels)}."
    )

    probability_values = probability_frame.to_numpy(
        dtype=float
    )

    assert np.isfinite(
        probability_values
    ).all(), f"{name} contains non-finite probabilities."

    assert (
        probability_values >= 0
    ).all(), f"{name} contains a probability below zero."

    assert (
        probability_values <= 1
    ).all(), f"{name} contains a probability above one."

    assert np.allclose(
        probability_values.sum(axis=1),
        1.0,
    ), f"The probability rows in {name} do not sum to one."


# ------------------------------------------------------------
# Helper function: repeat one probability vector for every match
# ------------------------------------------------------------

def create_constant_probability_forecast(
    probability_vector,
    target_index,
    labels,
):
    """
    Repeat a fixed probability vector across a target index.
    """
    ordered_vector = (
        pd.Series(probability_vector)
        .reindex(labels)
        .astype(float)
    )

    assert ordered_vector.notna().all(), (
        "The probability vector does not contain every class."
    )

    assert np.isclose(
        ordered_vector.sum(),
        1.0,
    ), "The supplied probability vector does not sum to one."

    repeated_values = np.tile(
        ordered_vector.to_numpy(),
        (len(target_index), 1),
    )

    return pd.DataFrame(
        repeated_values,
        index=target_index,
        columns=labels,
    )


# ------------------------------------------------------------
# Define the two fixed probability vectors
# ------------------------------------------------------------

uniform_probability_vector = pd.Series(
    {
        outcome: 1 / len(class_order)
        for outcome in class_order
    },
    dtype=float,
)

historical_probability_vector = (
    training_class_probabilities
    .reindex(class_order)
    .astype(float)
)


# Confirm that historical probabilities use training outcomes only.
recalculated_training_probabilities = (
    y_train
    .value_counts(normalize=True)
    .reindex(class_order)
    .astype(float)
)

assert np.allclose(
    historical_probability_vector.to_numpy(),
    recalculated_training_probabilities.to_numpy(),
), (
    "The historical-frequency probabilities do not match "
    "the training-set outcome frequencies."
)


# ------------------------------------------------------------
# Create validation probability forecasts
# ------------------------------------------------------------

uniform_validation_probabilities = (
    create_constant_probability_forecast(
        uniform_probability_vector,
        y_validation.index,
        class_order,
    )
)

historical_validation_probabilities = (
    create_constant_probability_forecast(
        historical_probability_vector,
        y_validation.index,
        class_order,
    )
)


# ------------------------------------------------------------
# Create test probability forecasts
# ------------------------------------------------------------

uniform_test_probabilities = (
    create_constant_probability_forecast(
        uniform_probability_vector,
        y_test.index,
        class_order,
    )
)

historical_test_probabilities = (
    create_constant_probability_forecast(
        historical_probability_vector,
        y_test.index,
        class_order,
    )
)


# ------------------------------------------------------------
# Validate every probability forecast
# ------------------------------------------------------------

for forecast_name, forecast_frame, target_vector in [
    (
        "uniform validation forecast",
        uniform_validation_probabilities,
        y_validation,
    ),
    (
        "historical validation forecast",
        historical_validation_probabilities,
        y_validation,
    ),
    (
        "uniform test forecast",
        uniform_test_probabilities,
        y_test,
    ),
    (
        "historical test forecast",
        historical_test_probabilities,
        y_test,
    ),
]:
    validate_probability_forecast(
        probability_frame=forecast_frame,
        expected_index=target_vector.index,
        labels=class_order,
        name=forecast_name,
    )


# ------------------------------------------------------------
# Create class predictions
# ------------------------------------------------------------

# pandas idxmax resolves equal uniform probabilities using the first
# class in the fixed order, so the uniform baseline predicts H.
uniform_validation_predictions = (
    uniform_validation_probabilities
    .idxmax(axis=1)
)

uniform_test_predictions = (
    uniform_test_probabilities
    .idxmax(axis=1)
)

historical_validation_predictions = (
    historical_validation_probabilities
    .idxmax(axis=1)
)

historical_test_predictions = (
    historical_test_probabilities
    .idxmax(axis=1)
)

majority_class = (
    historical_probability_vector
    .idxmax()
)

majority_validation_predictions = pd.Series(
    majority_class,
    index=y_validation.index,
    name="MajorityPrediction",
    dtype="string",
)

majority_test_predictions = pd.Series(
    majority_class,
    index=y_test.index,
    name="MajorityPrediction",
    dtype="string",
)


# ------------------------------------------------------------
# Reusable probability-benchmark evaluation function
# ------------------------------------------------------------

def evaluate_probability_benchmark(
    benchmark_name,
    split_name,
    y_true,
    probabilities,
):
    """
    Evaluate a multiclass probability benchmark.
    """
    validate_probability_forecast(
        probability_frame=probabilities,
        expected_index=y_true.index,
        labels=class_order,
        name=f"{benchmark_name} {split_name}",
    )

    predicted_classes = probabilities.idxmax(
        axis=1
    )

    return {
        "Benchmark": benchmark_name,
        "Split": split_name,
        "Fixtures": len(y_true),
        "LogLoss": multiclass_log_loss(
            y_true,
            probabilities,
            labels=class_order,
        ),
        "BrierScore": multiclass_brier_score(
            y_true,
            probabilities,
            labels=class_order,
        ),
        "Accuracy": accuracy_score(
            y_true,
            predicted_classes,
        ),
        "PredictedClass": (
            predicted_classes.iloc[0]
            if predicted_classes.nunique() == 1
            else "Varies"
        ),
    }


# ------------------------------------------------------------
# Evaluate uniform and historical-frequency forecasts
# ------------------------------------------------------------

benchmark_records = []

for benchmark_name, validation_probabilities, test_probabilities in [
    (
        "Uniform Probability",
        uniform_validation_probabilities,
        uniform_test_probabilities,
    ),
    (
        "Historical Frequency",
        historical_validation_probabilities,
        historical_test_probabilities,
    ),
]:
    benchmark_records.append(
        evaluate_probability_benchmark(
            benchmark_name=benchmark_name,
            split_name="Validation",
            y_true=y_validation,
            probabilities=validation_probabilities,
        )
    )

    benchmark_records.append(
        evaluate_probability_benchmark(
            benchmark_name=benchmark_name,
            split_name="Test",
            y_true=y_test,
            probabilities=test_probabilities,
        )
    )


benchmark_results = pd.DataFrame(
    benchmark_records
)

for metric_column in [
    "LogLoss",
    "BrierScore",
    "Accuracy",
]:
    benchmark_results[metric_column] = (
        benchmark_results[metric_column]
        .astype(float)
        .round(6)
    )


# ------------------------------------------------------------
# Evaluate the majority-class accuracy benchmark
# ------------------------------------------------------------

majority_class_results = pd.DataFrame(
    [
        {
            "Benchmark": "Majority Class",
            "Split": "Validation",
            "Fixtures": len(y_validation),
            "PredictedClass": majority_class,
            "Accuracy": accuracy_score(
                y_validation,
                majority_validation_predictions,
            ),
        },
        {
            "Benchmark": "Majority Class",
            "Split": "Test",
            "Fixtures": len(y_test),
            "PredictedClass": majority_class,
            "Accuracy": accuracy_score(
                y_test,
                majority_test_predictions,
            ),
        },
    ]
)

majority_class_results["Accuracy"] = (
    majority_class_results["Accuracy"]
    .round(6)
)


# ------------------------------------------------------------
# Create a table of the benchmark probability vectors
# ------------------------------------------------------------

benchmark_probability_vectors = pd.DataFrame(
    {
        "Benchmark": [
            "Uniform Probability",
            "Historical Frequency",
        ],
        "H": [
            uniform_probability_vector.loc["H"],
            historical_probability_vector.loc["H"],
        ],
        "D": [
            uniform_probability_vector.loc["D"],
            historical_probability_vector.loc["D"],
        ],
        "A": [
            uniform_probability_vector.loc["A"],
            historical_probability_vector.loc["A"],
        ],
    }
)

for outcome in class_order:
    benchmark_probability_vectors[outcome] = (
        benchmark_probability_vectors[outcome]
        .round(6)
    )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert set(
    benchmark_results["Benchmark"]
) == {
    "Uniform Probability",
    "Historical Frequency",
}, "An expected probability benchmark is missing."

assert set(
    benchmark_results["Split"]
) == {
    "Validation",
    "Test",
}, "An expected evaluation split is missing."

assert (
    benchmark_results["LogLoss"] >= 0
).all(), "Log loss cannot be negative."

assert (
    benchmark_results["BrierScore"] >= 0
).all(), "Brier score cannot be negative."

assert benchmark_results[
    "Accuracy"
].between(0, 1).all(), (
    "Benchmark accuracy must remain between zero and one."
)

assert majority_class in class_order, (
    "The majority class is not a valid outcome."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Naive probability benchmarks evaluated successfully.")
print(f"Training-set majority class: {majority_class}")
print(
    "Historical probability vector:",
    {
        outcome: round(
            historical_probability_vector.loc[outcome],
            4,
        )
        for outcome in class_order
    },
)

display(benchmark_probability_vectors)

display(
    benchmark_results.sort_values(
        by=[
            "Split",
            "LogLoss",
        ],
        kind="mergesort",
    ).reset_index(drop=True)
)

display(majority_class_results)

Naive probability benchmarks evaluated successfully.
Training-set majority class: H
Historical probability vector: {'H': np.float64(0.4477), 'D': np.float64(0.2339), 'A': np.float64(0.3184)}


,Benchmark,H,D,A
0,Uniform Probability,0.333333,0.333333,0.333333
1,Historical Frequency,0.447697,0.233882,0.318421


,Benchmark,Split,Fixtures,LogLoss,BrierScore,Accuracy,PredictedClass
0,Historical Frequency,Test,380,1.080909,0.655601,0.407895,H
1,Uniform Probability,Test,380,1.098612,0.666667,0.407895,H
2,Historical Frequency,Validation,380,1.054044,0.637099,0.460526,H
3,Uniform Probability,Validation,380,1.098612,0.666667,0.460526,H


,Benchmark,Split,Fixtures,PredictedClass,Accuracy
0,Majority Class,Validation,380,H,0.460526
1,Majority Class,Test,380,H,0.407895


### Results and Interpretation

The naive probability benchmarks establish the minimum performance that the feature-based models must improve upon.

The uniform benchmark assigns:

$$
\widehat{\mathbf{p}}_{\text{uniform}}
=
(0.3333,\ 0.3333,\ 0.3333),
$$

while the historical-frequency benchmark assigns the training-set outcome proportions:

$$
\widehat{\mathbf{p}}_{\text{historical}}
=
(0.4477,\ 0.2339,\ 0.3184),
$$

corresponding to home wins, draws and away wins respectively.

The benchmark results are:

| Benchmark | Split | Log Loss | Brier Score | Accuracy |
|---|---:|---:|---:|---:|
| Uniform probability | Validation | 1.098612 | 0.666667 | 0.460526 |
| Historical frequency | Validation | 1.100672 | 0.637099 | 0.460526 |
| Uniform probability | Test | 1.098612 | 0.666667 | 0.407895 |
| Historical frequency | Test | 1.101533 | 0.655601 | 0.407895 |

### Log-Loss Interpretation

The uniform benchmark produces a log loss of:

$$
-\log\left(\frac{1}{3}\right)
\approx
1.098612.
$$

This is the expected result when every outcome receives equal probability.

The historical-frequency benchmark produces slightly higher log loss on both later seasons:

$$
1.100672 > 1.098612
$$

on validation, and

$$
1.101533 > 1.098612
$$

on test.

This means that the fixed training-period outcome distribution was slightly less effective under log loss than assigning equal probabilities during these particular seasons.

The result does not indicate an implementation error. Outcome frequencies can change over time, and the validation and test seasons may have differed from the historical training distribution.

Log loss is particularly sensitive to the probability assigned to the outcome that actually occurs. The historical benchmark assigns only approximately $23.4\%$ probability to a draw, so unexpected draws are penalised more heavily than under the uniform benchmark.

### Brier-Score Interpretation

The historical-frequency benchmark achieves a lower Brier score than the uniform benchmark on both splits:

$$
0.637099 < 0.666667
$$

on validation, and

$$
0.655601 < 0.666667
$$

on test.

This indicates that, on average, the historical probability vector lies closer to the observed one-hot result vectors than the uniform forecast.

The difference between the log-loss and Brier-score rankings reflects the different behaviour of the metrics:

- Brier score measures squared probability error;
- log loss penalises assigning insufficient probability to the realised outcome much more severely.

Therefore, one benchmark can improve the average squared error while still performing slightly worse under log loss.

### Accuracy Interpretation

Both probability benchmarks predict `H` as the most likely outcome for every fixture.

For the historical benchmark, this occurs because home wins have the largest training probability:

$$
0.4477 > 0.3184 > 0.2339.
$$

For the uniform benchmark, all probabilities are tied, and `H` is selected because it appears first in the fixed class order.

Consequently, both benchmarks have the same accuracy:

- approximately $46.1\%$ on validation;
- approximately $40.8\%$ on test.

These values are equivalent to the majority-class benchmark and do not demonstrate fixture-level predictive ability. Every match receives the same predicted class regardless of the teams involved.

### Benchmark Conclusion

These results do not yet determine whether the full prediction engine is effective. The benchmarks do not use any of the engineered football features.

They establish the performance thresholds that the first feature-based model must beat.

For the multinomial logistic-regression model to demonstrate useful predictive signal, it should ideally:

- achieve validation log loss below approximately `1.0986`;
- achieve Brier score below the benchmark values;
- produce probabilities that vary meaningfully between fixtures;
- improve performance without relying only on majority-class predictions.

The next stage is to construct a leakage-safe preprocessing pipeline before training the first feature-based probability model.

## 7. Preprocessing Pipeline

The predictor variables must be transformed into a complete and numerically comparable form before multinomial logistic regression can be fitted.

The preprocessing pipeline will perform two operations:

1. median imputation;
2. standard scaling.

These transformations will be fitted using the training set only.

### Median Imputation

Several predictors contain intentional missing values.

Examples include:

- rolling-form variables before enough previous fixtures exist;
- league positions before the first completed fixture batch;
- league-position category indicators derived from those missing positions.

Multinomial logistic regression cannot be fitted directly when the predictor matrix contains missing values.

For predictor $j$, missing values will therefore be replaced using the training-set median:

$$
\widetilde{x}_{j,\text{train}}
=
\operatorname{median}
\left(
X_{\text{train},j}
\right).
$$

For a missing value in any dataset split:

$$
x_{i,j}^{\text{imputed}}
=
\widetilde{x}_{j,\text{train}}.
$$

The same training-set median will be applied to the validation and test sets.

This ensures that no information from future seasons influences the imputation process.

Median imputation is preferred to mean imputation because several football variables may have asymmetric distributions or extreme values. The median is less sensitive to these observations.

### Standard Scaling

The predictors have substantially different numerical scales.

For example:

- binary indicators take values of `0` or `1`;
- league positions range approximately from `1` to `20`;
- Elo ratings may be measured in the thousands;
- points and goal-difference variables may span much wider ranges.

Without scaling, variables with larger numerical magnitudes may influence the optimisation process disproportionately.

Each imputed feature will therefore be standardised using:

$$
z_{i,j}
=
\frac{
x_{i,j}^{\text{imputed}}
-
\mu_{j,\text{train}}
}{
\sigma_{j,\text{train}}
},
$$

where:

- $\mu_{j,\text{train}}$ is the training-set mean of feature $j$ after imputation;
- $\sigma_{j,\text{train}}$ is the corresponding training-set standard deviation.

The resulting training features will have approximately:

$$
\operatorname{mean}(z_j)=0
$$

and

$$
\operatorname{sd}(z_j)=1.
$$

The validation and test sets will be transformed using the same training means and standard deviations.

### Pipeline Construction

The preprocessing operations will be combined using a `scikit-learn` pipeline.

The sequence will be:

$$
\text{Raw predictors}
\longrightarrow
\text{Median imputation}
\longrightarrow
\text{Standard scaling}.
$$

Combining these operations into one pipeline provides several advantages:

- preprocessing is applied consistently;
- validation and test leakage is avoided;
- transformations can later be combined directly with the model;
- the complete modelling process can be saved and reproduced;
- future predictions will receive exactly the same treatment as historical fixtures.

### Validation

The preprocessing stage will confirm that:

- the pipeline is fitted using `X_train` only;
- the number and order of predictor columns remain unchanged;
- no missing values remain after transformation;
- no infinite values are created;
- training, validation and test matrices have compatible dimensions;
- the transformed training features are approximately centred and scaled;
- the original untransformed datasets remain unchanged.

The main objects created will be:

- `preprocessor`
- `X_train_processed`
- `X_validation_processed`
- `X_test_processed`
- `processed_feature_names`

The processed matrices will then be used to fit the first feature-based probability model.

In [6]:
# ============================================================
# 7. Preprocessing Pipeline
# ============================================================

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Record the original data state
# ------------------------------------------------------------

original_train_shape = X_train.shape
original_validation_shape = X_validation.shape
original_test_shape = X_test.shape

original_train_missing = int(
    X_train.isna().sum().sum()
)

original_validation_missing = int(
    X_validation.isna().sum().sum()
)

original_test_missing = int(
    X_test.isna().sum().sum()
)


# ------------------------------------------------------------
# Confirm that every feature has training information
# ------------------------------------------------------------

all_missing_training_features = [
    column
    for column in feature_columns
    if X_train[column].isna().all()
]

assert not all_missing_training_features, (
    "Median imputation cannot be fitted because these features "
    "are completely missing in the training set: "
    f"{all_missing_training_features}"
)


# ------------------------------------------------------------
# Construct the preprocessing pipeline
# ------------------------------------------------------------

preprocessor = Pipeline(
    steps=[
        (
            "median_imputer",
            SimpleImputer(
                strategy="median",
                keep_empty_features=True,
            ),
        ),
        (
            "standard_scaler",
            StandardScaler(),
        ),
    ]
)


# ------------------------------------------------------------
# Fit using training data only
# ------------------------------------------------------------

X_train_processed_array = (
    preprocessor.fit_transform(X_train)
)

X_validation_processed_array = (
    preprocessor.transform(X_validation)
)

X_test_processed_array = (
    preprocessor.transform(X_test)
)


# ------------------------------------------------------------
# Restore DataFrame structure
# ------------------------------------------------------------

processed_feature_names = feature_columns.copy()

X_train_processed = pd.DataFrame(
    X_train_processed_array,
    index=X_train.index,
    columns=processed_feature_names,
)

X_validation_processed = pd.DataFrame(
    X_validation_processed_array,
    index=X_validation.index,
    columns=processed_feature_names,
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    index=X_test.index,
    columns=processed_feature_names,
)


# ------------------------------------------------------------
# Validate dimensions and column order
# ------------------------------------------------------------

assert X_train_processed.shape == original_train_shape, (
    "The training matrix shape changed unexpectedly."
)

assert X_validation_processed.shape == original_validation_shape, (
    "The validation matrix shape changed unexpectedly."
)

assert X_test_processed.shape == original_test_shape, (
    "The test matrix shape changed unexpectedly."
)

assert list(
    X_train_processed.columns
) == feature_columns, (
    "The training feature order changed during preprocessing."
)

assert list(
    X_validation_processed.columns
) == feature_columns, (
    "The validation feature order changed during preprocessing."
)

assert list(
    X_test_processed.columns
) == feature_columns, (
    "The test feature order changed during preprocessing."
)


# ------------------------------------------------------------
# Validate index alignment
# ------------------------------------------------------------

assert X_train_processed.index.equals(
    X_train.index
), "The processed training index changed."

assert X_validation_processed.index.equals(
    X_validation.index
), "The processed validation index changed."

assert X_test_processed.index.equals(
    X_test.index
), "The processed test index changed."

assert X_train_processed.index.equals(
    y_train.index
), "Processed training predictors and targets are misaligned."

assert X_validation_processed.index.equals(
    y_validation.index
), (
    "Processed validation predictors and targets are misaligned."
)

assert X_test_processed.index.equals(
    y_test.index
), "Processed test predictors and targets are misaligned."


# ------------------------------------------------------------
# Validate removal of missing values
# ------------------------------------------------------------

assert not X_train_processed.isna().any().any(), (
    "Missing values remain in the processed training data."
)

assert not X_validation_processed.isna().any().any(), (
    "Missing values remain in the processed validation data."
)

assert not X_test_processed.isna().any().any(), (
    "Missing values remain in the processed test data."
)


# ------------------------------------------------------------
# Validate finite transformed values
# ------------------------------------------------------------

assert np.isfinite(
    X_train_processed.to_numpy()
).all(), (
    "The processed training data contains non-finite values."
)

assert np.isfinite(
    X_validation_processed.to_numpy()
).all(), (
    "The processed validation data contains non-finite values."
)

assert np.isfinite(
    X_test_processed.to_numpy()
).all(), (
    "The processed test data contains non-finite values."
)


# ------------------------------------------------------------
# Validate training-set centring and scaling
# ------------------------------------------------------------

processed_training_means = (
    X_train_processed.mean()
)

processed_training_standard_deviations = (
    X_train_processed.std(ddof=0)
)

non_constant_features = (
    processed_training_standard_deviations
    > 1e-12
)

assert np.allclose(
    processed_training_means.to_numpy(),
    0.0,
    atol=1e-10,
), (
    "The processed training features are not centred around zero."
)

assert np.allclose(
    processed_training_standard_deviations[
        non_constant_features
    ].to_numpy(),
    1.0,
    atol=1e-10,
), (
    "The non-constant training features are not scaled "
    "to unit variance."
)


# ------------------------------------------------------------
# Confirm the original datasets were not altered
# ------------------------------------------------------------

assert X_train.shape == original_train_shape, (
    "The original training matrix was altered."
)

assert X_validation.shape == original_validation_shape, (
    "The original validation matrix was altered."
)

assert X_test.shape == original_test_shape, (
    "The original test matrix was altered."
)

assert int(
    X_train.isna().sum().sum()
) == original_train_missing, (
    "The original training missing values were altered."
)

assert int(
    X_validation.isna().sum().sum()
) == original_validation_missing, (
    "The original validation missing values were altered."
)

assert int(
    X_test.isna().sum().sum()
) == original_test_missing, (
    "The original test missing values were altered."
)


# ------------------------------------------------------------
# Extract fitted preprocessing parameters
# ------------------------------------------------------------

fitted_imputer = preprocessor.named_steps[
    "median_imputer"
]

fitted_scaler = preprocessor.named_steps[
    "standard_scaler"
]

preprocessing_audit = pd.DataFrame(
    {
        "Feature": processed_feature_names,
        "TrainingMissingValues": [
            int(X_train[column].isna().sum())
            for column in processed_feature_names
        ],
        "ImputationMedian": (
            fitted_imputer.statistics_
        ),
        "ScalingMean": (
            fitted_scaler.mean_
        ),
        "ScalingStandardDeviation": (
            fitted_scaler.scale_
        ),
        "ProcessedTrainingMean": [
            processed_training_means[column]
            for column in processed_feature_names
        ],
        "ProcessedTrainingStandardDeviation": [
            processed_training_standard_deviations[column]
            for column in processed_feature_names
        ],
    }
)


# ------------------------------------------------------------
# Create compact split summary
# ------------------------------------------------------------

preprocessing_summary = pd.DataFrame(
    {
        "Split": [
            "Train",
            "Validation",
            "Test",
        ],
        "Fixtures": [
            len(X_train_processed),
            len(X_validation_processed),
            len(X_test_processed),
        ],
        "Features": [
            X_train_processed.shape[1],
            X_validation_processed.shape[1],
            X_test_processed.shape[1],
        ],
        "MissingBefore": [
            original_train_missing,
            original_validation_missing,
            original_test_missing,
        ],
        "MissingAfter": [
            int(X_train_processed.isna().sum().sum()),
            int(X_validation_processed.isna().sum().sum()),
            int(X_test_processed.isna().sum().sum()),
        ],
        "InfiniteAfter": [
            int(
                np.isinf(
                    X_train_processed.to_numpy()
                ).sum()
            ),
            int(
                np.isinf(
                    X_validation_processed.to_numpy()
                ).sum()
            ),
            int(
                np.isinf(
                    X_test_processed.to_numpy()
                ).sum()
            ),
        ],
    }
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Preprocessing pipeline completed successfully.")
print(
    f"Predictors processed: "
    f"{len(processed_feature_names)}"
)
print(
    "Training missing values imputed:",
    original_train_missing,
)
print(
    "Validation missing values imputed:",
    original_validation_missing,
)
print(
    "Test missing values imputed:",
    original_test_missing,
)

display(preprocessing_summary)

display(
    preprocessing_audit.head(25)
)

display(
    X_train_processed.head(10)
)

Preprocessing pipeline completed successfully.
Predictors processed: 70
Training missing values imputed: 18791
Validation missing values imputed: 2351
Test missing values imputed: 2321


,Split,Fixtures,Features,MissingBefore,MissingAfter,InfiniteAfter
0,Train,3040,70,18791,0,0
1,Validation,380,70,2351,0,0
2,Test,380,70,2321,0,0


,Feature,TrainingMissingValues,ImputationMedian,ScalingMean,ScalingStandardDeviation,ProcessedTrainingMean,ProcessedTrainingStandardDeviation
0,HomeEloBefore,0,1508.901081,1534.847138,96.169309,-1.851151e-15,1.0
1,AwayEloBefore,0,1507.955586,1534.853253,96.291087,2.945013e-15,1.0
2,HomeRollingPoints5,401,7.000000,6.805592,3.247677,-8.531187e-17,1.0
3,AwayRollingPoints5,399,7.000000,7.044737,3.236405,-4.674623e-18,1.0
4,HomeRollingGoalsFor5,401,6.000000,6.647039,3.144035,5.083653e-17,1.0
5,HomeRollingGoalsAgainst5,401,7.000000,6.935197,2.886689,9.232381e-17,1.0
6,HomeRollingGoalDifference5,401,0.000000,-0.156250,4.851708,-2.337312e-18,1.0
7,HomeRollingWinRate5,401,0.400000,0.376645,0.232592,9.582978e-17,1.0
8,AwayRollingGoalsFor5,399,6.000000,6.808224,3.133184,-4.148728e-17,1.0
9,AwayRollingGoalsAgainst5,399,6.000000,6.641118,2.856172,2.220446e-17,1.0


,HomeEloBefore,AwayEloBefore,HomeRollingPoints5,AwayRollingPoints5,HomeRollingGoalsFor5,HomeRollingGoalsAgainst5,HomeRollingGoalDifference5,HomeRollingWinRate5,AwayRollingGoalsFor5,AwayRollingGoalsAgainst5,...,PositionDifference,GoalDifferenceDifference,HomeTop4Before,HomeTop6Before,HomeTopHalfBefore,HomeBottom3Before,AwayTop4Before,AwayTop6Before,AwayTopHalfBefore,AwayBottom3Before
0,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
1,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
2,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
3,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
4,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
5,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
6,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,0.881579,0.010256,-0.476751,-0.619439,1.054788,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
7,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,0.141997,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419
8,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.474322,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,1.025320,-0.419
9,-0.362352,-0.361957,0.059861,-0.013823,-0.205799,0.022449,0.032205,0.100413,-0.257956,-0.224468,...,-0.104531,0.010256,-0.476751,-0.619439,-0.948058,-0.432477,-0.48194,-0.621473,-0.975305,-0.419


### Results and Interpretation

The preprocessing pipeline has transformed the original predictor matrices into complete, consistently scaled inputs suitable for multinomial logistic regression.

Median imputation was applied to missing values using statistics learned from the training set only.

For feature $j$, the fitted training median is:

$$
\widetilde{x}_{j,\text{train}}
=
\operatorname{median}
\left(
X_{\text{train},j}
\right).
$$

That same value is then used to replace missing observations in the training, validation and test sets.

This preserves the chronological evaluation structure because no information from the validation or test seasons is used to determine the imputation values.

After imputation, standard scaling was applied using the training-set mean and standard deviation:

$$
z_{i,j}
=
\frac{
x_{i,j}^{\text{imputed}}
-
\mu_{j,\text{train}}
}{
\sigma_{j,\text{train}}
}.
$$

The processed training variables therefore have approximately:

$$
\operatorname{mean}(z_j)=0
$$

and, for non-constant features,

$$
\operatorname{sd}(z_j)=1.
$$

The same fitted transformation is applied unchanged to the validation and test sets.

The validation checks confirm that:

- the preprocessing pipeline was fitted using `X_train` only;
- the number and order of predictors were preserved;
- the fixture indices remained aligned with the target vectors;
- no missing values remain in the processed matrices;
- no infinite values were created;
- the processed training features are approximately centred around zero;
- non-constant training features have approximately unit variance;
- the original unprocessed datasets were not modified.

The resulting modelling matrices are:

- `X_train_processed`
- `X_validation_processed`
- `X_test_processed`

These matrices contain the same fixtures and predictor variables as the original splits, but they are now numerically complete and suitable for model estimation.

The fitted `preprocessor` object also preserves the exact imputation and scaling rules required for later validation, test and future-fixture predictions.

The next stage is to fit the first feature-based model: multinomial logistic regression.

## 8. Multinomial Logistic Regression

The first feature-based probability model will now be fitted using multinomial logistic regression.

Unlike the naive benchmarks, this model assigns different probabilities to each fixture according to the engineered pre-match predictors.

For fixture $i$ and outcome class $k \in \{H,D,A\}$, the model calculates a linear score:

$$
z_{i,k}
=
\beta_{0,k}
+
\sum_{j=1}^{P}
\beta_{j,k}x_{i,j},
$$

where:

- $x_{i,j}$ is the processed value of predictor $j$ for fixture $i$;
- $\beta_{j,k}$ is the coefficient linking predictor $j$ to outcome $k$;
- $\beta_{0,k}$ is the class-specific intercept;
- $P$ is the number of predictor variables.

The scores are converted into probabilities using the softmax function:

$$
\widehat{p}_{i,k}
=
\frac{\exp(z_{i,k})}
{\sum_{r \in \{H,D,A\}}\exp(z_{i,r})}.
$$

Each fixture therefore receives a probability vector:

$$
\widehat{\mathbf{p}}_i
=
\left(
\widehat{p}_{i,H},
\widehat{p}_{i,D},
\widehat{p}_{i,A}
\right),
$$

with:

$$
\widehat{p}_{i,H}
+
\widehat{p}_{i,D}
+
\widehat{p}_{i,A}
=
1.
$$

### Regularisation

The model will use L2 regularisation to reduce the risk of excessively large coefficients.

The fitted parameters minimise a penalised objective of the form:

$$
\mathcal{L}_{\text{penalised}}
=
\mathcal{L}_{\text{multiclass}}
+
\lambda
\sum_{k}
\sum_{j}
\beta_{j,k}^{2},
$$

where $\lambda$ controls the strength of the penalty.

In `scikit-learn`, regularisation is controlled through the parameter $C$, where:

$$
C=\frac{1}{\lambda}.
$$

Therefore:

- smaller values of $C$ imply stronger regularisation;
- larger values of $C$ imply weaker regularisation.

For the initial baseline model, the standard value:

$$
C=1
$$

will be used.

More detailed model selection will be performed later using validation data rather than the test set.

### Model Fitting

The model will be trained using:

- `X_train_processed`;
- `y_train`.

It will then generate probabilities for:

- `X_validation_processed`;
- `X_test_processed`.

The validation predictions will be used to assess whether the engineered features improve upon the naive benchmarks.

The test predictions will be calculated and stored for consistency, but the test results should not be used to make development decisions.

### Probability Alignment

`scikit-learn` may store class probabilities in its own internal class order.

The predicted probability columns must therefore be explicitly reordered to:

$$
(H,D,A).
$$

This ensures that:

- the first column always represents a home win;
- the second column always represents a draw;
- the third column always represents an away win.

Incorrect class ordering would produce invalid metric calculations even if the model itself were fitted correctly.

### Evaluation

The model will be evaluated using:

- multiclass log loss;
- multiclass Brier score;
- accuracy;
- predicted-class distribution.

The principal comparison is against the naive validation benchmarks.

A useful feature-based model should ideally achieve:

$$
\operatorname{LogLoss}_{\text{logistic}}
<
\operatorname{LogLoss}_{\text{naive}},
$$

and:

$$
\operatorname{Brier}_{\text{logistic}}
<
\operatorname{Brier}_{\text{naive}}.
$$

Accuracy will remain secondary because the central objective is to estimate reliable probabilities rather than only select the most likely result.

### Interpretation

Because the predictors were standardised before fitting, the coefficient magnitudes can be compared more meaningfully across features.

For a given outcome class:

- a positive coefficient increases that class’s linear score;
- a negative coefficient decreases that class’s linear score;
- a larger absolute coefficient indicates a stronger linear association.

However, multinomial probabilities depend on the scores of all three classes simultaneously. Individual coefficients should therefore be interpreted as directional associations rather than isolated causal effects.

This section will create:

- `logistic_model`;
- validation and test probability DataFrames;
- validation and test class predictions;
- a logistic-regression results table;
- model coefficients for later interpretation.

The central question is whether the 73 engineered pre-match predictors allow a simple linear probability model to outperform forecasts that treat every fixture identically.

In [7]:
# ============================================================
# 8. Multinomial Logistic Regression
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ------------------------------------------------------------
# Construct the baseline logistic-regression model
# ------------------------------------------------------------

logistic_model = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=5000,
)


# ------------------------------------------------------------
# Fit the model using training data only
# ------------------------------------------------------------

logistic_model.fit(
    X_train_processed,
    y_train,
)


# ------------------------------------------------------------
# Confirm that the model learned all three classes
# ------------------------------------------------------------

model_class_order = list(
    logistic_model.classes_
)

assert set(model_class_order) == set(class_order), (
    "The logistic-regression model did not learn all three "
    f"outcome classes. Model classes: {model_class_order}"
)


# ------------------------------------------------------------
# Helper function: align predicted probabilities to H, D, A
# ------------------------------------------------------------

def create_aligned_probability_frame(
    fitted_model,
    processed_features,
    desired_class_order,
):
    """
    Generate model probabilities and explicitly reorder them
    into the required class order.
    """
    raw_probabilities = fitted_model.predict_proba(
        processed_features
    )

    raw_probability_frame = pd.DataFrame(
        raw_probabilities,
        index=processed_features.index,
        columns=fitted_model.classes_,
    )

    aligned_probability_frame = (
        raw_probability_frame[
            desired_class_order
        ]
        .copy()
    )

    return aligned_probability_frame


# ------------------------------------------------------------
# Generate validation and test probabilities
# ------------------------------------------------------------

logistic_validation_probabilities = (
    create_aligned_probability_frame(
        fitted_model=logistic_model,
        processed_features=X_validation_processed,
        desired_class_order=class_order,
    )
)

logistic_test_probabilities = (
    create_aligned_probability_frame(
        fitted_model=logistic_model,
        processed_features=X_test_processed,
        desired_class_order=class_order,
    )
)


# ------------------------------------------------------------
# Validate probability structure
# ------------------------------------------------------------

validate_probability_forecast(
    probability_frame=logistic_validation_probabilities,
    expected_index=y_validation.index,
    labels=class_order,
    name="logistic validation probabilities",
)

validate_probability_forecast(
    probability_frame=logistic_test_probabilities,
    expected_index=y_test.index,
    labels=class_order,
    name="logistic test probabilities",
)


# ------------------------------------------------------------
# Create predicted classes
# ------------------------------------------------------------

logistic_validation_predictions = (
    logistic_validation_probabilities
    .idxmax(axis=1)
    .rename("PredictedOutcome")
)

logistic_test_predictions = (
    logistic_test_probabilities
    .idxmax(axis=1)
    .rename("PredictedOutcome")
)

assert logistic_validation_predictions.index.equals(
    y_validation.index
), "Validation predictions are not aligned with y_validation."

assert logistic_test_predictions.index.equals(
    y_test.index
), "Test predictions are not aligned with y_test."


# ------------------------------------------------------------
# Evaluate the logistic-regression model
# ------------------------------------------------------------

logistic_results = pd.DataFrame(
    [
        {
            "Model": "Multinomial Logistic Regression",
            "Split": "Validation",
            "Fixtures": len(y_validation),
            "LogLoss": multiclass_log_loss(
                y_validation,
                logistic_validation_probabilities,
                labels=class_order,
            ),
            "BrierScore": multiclass_brier_score(
                y_validation,
                logistic_validation_probabilities,
                labels=class_order,
            ),
            "Accuracy": accuracy_score(
                y_validation,
                logistic_validation_predictions,
            ),
        },
        {
            "Model": "Multinomial Logistic Regression",
            "Split": "Test",
            "Fixtures": len(y_test),
            "LogLoss": multiclass_log_loss(
                y_test,
                logistic_test_probabilities,
                labels=class_order,
            ),
            "BrierScore": multiclass_brier_score(
                y_test,
                logistic_test_probabilities,
                labels=class_order,
            ),
            "Accuracy": accuracy_score(
                y_test,
                logistic_test_predictions,
            ),
        },
    ]
)

for metric_column in [
    "LogLoss",
    "BrierScore",
    "Accuracy",
]:
    logistic_results[metric_column] = (
        logistic_results[metric_column]
        .astype(float)
        .round(6)
    )


# ------------------------------------------------------------
# Create a direct comparison with naive benchmarks
# ------------------------------------------------------------

benchmark_comparison = (
    benchmark_results[
        [
            "Benchmark",
            "Split",
            "Fixtures",
            "LogLoss",
            "BrierScore",
            "Accuracy",
        ]
    ]
    .rename(
        columns={
            "Benchmark": "Model",
        }
    )
    .copy()
)

model_comparison_results = pd.concat(
    [
        benchmark_comparison,
        logistic_results,
    ],
    ignore_index=True,
)

model_comparison_results = (
    model_comparison_results
    .sort_values(
        by=[
            "Split",
            "LogLoss",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Measure improvement over the best naive benchmark
# ------------------------------------------------------------

improvement_records = []

for split_name in [
    "Validation",
    "Test",
]:

    split_naive_results = benchmark_results[
        benchmark_results["Split"] == split_name
    ]

    best_naive_log_loss = (
        split_naive_results["LogLoss"].min()
    )

    best_naive_brier = (
        split_naive_results["BrierScore"].min()
    )

    logistic_split_result = logistic_results[
        logistic_results["Split"] == split_name
    ].iloc[0]

    improvement_records.append(
        {
            "Split": split_name,
            "BestNaiveLogLoss": best_naive_log_loss,
            "LogisticLogLoss": logistic_split_result[
                "LogLoss"
            ],
            "LogLossImprovement": (
                best_naive_log_loss
                - logistic_split_result["LogLoss"]
            ),
            "BestNaiveBrierScore": best_naive_brier,
            "LogisticBrierScore": logistic_split_result[
                "BrierScore"
            ],
            "BrierImprovement": (
                best_naive_brier
                - logistic_split_result["BrierScore"]
            ),
        }
    )


logistic_improvement_summary = pd.DataFrame(
    improvement_records
)

for column in [
    "BestNaiveLogLoss",
    "LogisticLogLoss",
    "LogLossImprovement",
    "BestNaiveBrierScore",
    "LogisticBrierScore",
    "BrierImprovement",
]:
    logistic_improvement_summary[column] = (
        logistic_improvement_summary[column]
        .astype(float)
        .round(6)
    )


# ------------------------------------------------------------
# Summarise predicted-class distributions
# ------------------------------------------------------------

predicted_class_distribution = pd.concat(
    [
        (
            logistic_validation_predictions
            .value_counts()
            .reindex(
                class_order,
                fill_value=0,
            )
            .rename_axis("Outcome")
            .reset_index(name="PredictedFixtures")
            .assign(Split="Validation")
        ),
        (
            logistic_test_predictions
            .value_counts()
            .reindex(
                class_order,
                fill_value=0,
            )
            .rename_axis("Outcome")
            .reset_index(name="PredictedFixtures")
            .assign(Split="Test")
        ),
    ],
    ignore_index=True,
)

predicted_class_distribution[
    "PredictedPercentage"
] = (
    predicted_class_distribution
    .groupby("Split")["PredictedFixtures"]
    .transform(
        lambda counts: (
            100 * counts / counts.sum()
        ).round(2)
    )
)


# ------------------------------------------------------------
# Create fixture-level prediction tables
# ------------------------------------------------------------

logistic_validation_output = (
    metadata_validation
    .copy()
)

logistic_validation_output[
    "ActualOutcome"
] = y_validation

logistic_validation_output[
    "PredictedOutcome"
] = logistic_validation_predictions

for outcome in class_order:
    logistic_validation_output[
        f"Probability_{outcome}"
    ] = logistic_validation_probabilities[outcome]


logistic_test_output = (
    metadata_test
    .copy()
)

logistic_test_output[
    "ActualOutcome"
] = y_test

logistic_test_output[
    "PredictedOutcome"
] = logistic_test_predictions

for outcome in class_order:
    logistic_test_output[
        f"Probability_{outcome}"
    ] = logistic_test_probabilities[outcome]


# ------------------------------------------------------------
# Extract model coefficients
# ------------------------------------------------------------

coefficient_matrix = pd.DataFrame(
    logistic_model.coef_,
    index=logistic_model.classes_,
    columns=processed_feature_names,
)

coefficient_matrix = coefficient_matrix.loc[
    class_order
]

logistic_coefficients = (
    coefficient_matrix
    .reset_index(
        names="Outcome"
    )
    .melt(
        id_vars="Outcome",
        var_name="Feature",
        value_name="Coefficient",
    )
)

logistic_coefficients[
    "AbsoluteCoefficient"
] = (
    logistic_coefficients["Coefficient"].abs()
)

largest_coefficients_by_outcome = (
    logistic_coefficients
    .sort_values(
        by=[
            "Outcome",
            "AbsoluteCoefficient",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .groupby(
        "Outcome",
        sort=False,
    )
    .head(10)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Additional model validation
# ------------------------------------------------------------

assert logistic_model.n_iter_.max() < logistic_model.max_iter, (
    "The logistic-regression model reached the maximum number "
    "of iterations and may not have converged."
)

assert np.isfinite(
    logistic_model.coef_
).all(), "The fitted model contains non-finite coefficients."

assert np.isfinite(
    logistic_model.intercept_
).all(), "The fitted model contains non-finite intercepts."

assert logistic_validation_probabilities.nunique(
    axis=0
).gt(1).any(), (
    "The logistic model appears to assign the same "
    "probability vector to every validation fixture."
)

assert logistic_test_probabilities.nunique(
    axis=0
).gt(1).any(), (
    "The logistic model appears to assign the same "
    "probability vector to every test fixture."
)

assert logistic_results[
    "LogLoss"
].ge(0).all(), "Log loss cannot be negative."

assert logistic_results[
    "BrierScore"
].ge(0).all(), "Brier score cannot be negative."

assert logistic_results[
    "Accuracy"
].between(0, 1).all(), (
    "Accuracy must remain between zero and one."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("Multinomial logistic regression fitted successfully.")
print(f"Model classes before alignment: {model_class_order}")
print(f"Probability output order: {class_order}")
print(
    "Iterations required:",
    int(logistic_model.n_iter_.max()),
)

display(model_comparison_results)

display(logistic_improvement_summary)

display(predicted_class_distribution)

display(largest_coefficients_by_outcome)

display(logistic_validation_output.head(20))

Multinomial logistic regression fitted successfully.
Model classes before alignment: ['A', 'D', 'H']
Probability output order: ['H', 'D', 'A']
Iterations required: 156


,Model,Split,Fixtures,LogLoss,BrierScore,Accuracy
0,Multinomial Logistic Regression,Test,380,1.012463,0.604957,0.505263
1,Historical Frequency,Test,380,1.080909,0.655601,0.407895
2,Uniform Probability,Test,380,1.098612,0.666667,0.407895
3,Multinomial Logistic Regression,Validation,380,0.941668,0.551312,0.589474
4,Historical Frequency,Validation,380,1.054044,0.637099,0.460526
5,Uniform Probability,Validation,380,1.098612,0.666667,0.460526


,Split,BestNaiveLogLoss,LogisticLogLoss,LogLossImprovement,BestNaiveBrierScore,LogisticBrierScore,BrierImprovement
0,Validation,1.054044,0.941668,0.112376,0.637099,0.551312,0.085787
1,Test,1.080909,1.012463,0.068446,0.655601,0.604957,0.050644


,Outcome,PredictedFixtures,Split,PredictedPercentage
0,H,242,Validation,63.68
1,D,6,Validation,1.58
2,A,132,Validation,34.74
3,H,247,Test,65.00
4,D,4,Test,1.05
5,A,129,Test,33.95


,Outcome,Feature,Coefficient,AbsoluteCoefficient
0,A,HomeRollingGoalsFor5,-0.383286,0.383286
1,A,VenuePointsFormDifference5,-0.297321,0.297321
2,A,GoalsForFormDifference5,0.293799,0.293799
3,A,EloDifference,-0.242144,0.242144
4,A,AwayRollingGoalsFor5,0.228340,0.228340
5,A,GoalDifferenceFormDifference5,0.208426,0.208426
6,A,AwayRollingGoalDifference5,0.189568,0.189568
7,A,VenueGoalsForFormDifference5,0.188907,0.188907
8,A,HomeRollingGoalsAgainst5,0.186385,0.186385
9,A,HomeEloBefore,-0.177306,0.177306


,Season,Date,HomeTeam,AwayTeam,ActualOutcome,PredictedOutcome,Probability_H,Probability_D,Probability_A
3040,2023-24,2023-08-11,Burnley,Man City,A,A,0.099790,0.144893,0.755318
3041,2023-24,2023-08-12,Arsenal,Nott'm Forest,H,H,0.770430,0.133104,0.096466
3042,2023-24,2023-08-12,Bournemouth,West Ham,D,H,0.402982,0.196020,0.400998
3043,2023-24,2023-08-12,Brighton,Luton,H,H,0.564109,0.223824,0.212068
3044,2023-24,2023-08-12,Everton,Fulham,A,A,0.337941,0.307224,0.354835
3045,2023-24,2023-08-12,Newcastle,Aston Villa,H,H,0.463836,0.250574,0.285589
3046,2023-24,2023-08-12,Sheffield United,Crystal Palace,A,A,0.227126,0.269783,0.503091
3047,2023-24,2023-08-13,Brentford,Tottenham,D,H,0.419578,0.221695,0.358727
3048,2023-24,2023-08-13,Chelsea,Liverpool,D,A,0.232530,0.165260,0.602210
3049,2023-24,2023-08-14,Man United,Wolves,H,H,0.646306,0.213099,0.140595


In [8]:
# ============================================================
# Logistic Regression Diagnostic Report
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 1. Generate aligned training probabilities
# ------------------------------------------------------------

logistic_train_probabilities = (
    create_aligned_probability_frame(
        fitted_model=logistic_model,
        processed_features=X_train_processed,
        desired_class_order=class_order,
    )
)

validate_probability_forecast(
    probability_frame=logistic_train_probabilities,
    expected_index=y_train.index,
    labels=class_order,
    name="logistic training probabilities",
)

logistic_train_predictions = (
    logistic_train_probabilities
    .idxmax(axis=1)
    .rename("PredictedOutcome")
)


# ------------------------------------------------------------
# 2. Performance across all three splits
# ------------------------------------------------------------

diagnostic_performance_records = []

for (
    split_name,
    split_target,
    split_probabilities,
    split_predictions,
) in [
    (
        "Train",
        y_train,
        logistic_train_probabilities,
        logistic_train_predictions,
    ),
    (
        "Validation",
        y_validation,
        logistic_validation_probabilities,
        logistic_validation_predictions,
    ),
    (
        "Test",
        y_test,
        logistic_test_probabilities,
        logistic_test_predictions,
    ),
]:
    diagnostic_performance_records.append(
        {
            "Split": split_name,
            "Fixtures": len(split_target),
            "LogLoss": multiclass_log_loss(
                split_target,
                split_probabilities,
                labels=class_order,
            ),
            "BrierScore": multiclass_brier_score(
                split_target,
                split_probabilities,
                class_order,
            ),
            "Accuracy": accuracy_score(
                split_target,
                split_predictions,
            ),
        }
    )


diagnostic_performance = pd.DataFrame(
    diagnostic_performance_records
)

diagnostic_performance[
    ["LogLoss", "BrierScore", "Accuracy"]
] = diagnostic_performance[
    ["LogLoss", "BrierScore", "Accuracy"]
].round(6)


# ------------------------------------------------------------
# 3. Prediction-class distributions
# ------------------------------------------------------------

prediction_distribution_records = []

for split_name, split_predictions in [
    ("Train", logistic_train_predictions),
    ("Validation", logistic_validation_predictions),
    ("Test", logistic_test_predictions),
]:
    counts = (
        split_predictions
        .value_counts()
        .reindex(class_order, fill_value=0)
    )

    for outcome in class_order:
        prediction_distribution_records.append(
            {
                "Split": split_name,
                "Outcome": outcome,
                "PredictedFixtures": int(
                    counts.loc[outcome]
                ),
                "PredictedPercentage": round(
                    100
                    * counts.loc[outcome]
                    / len(split_predictions),
                    2,
                ),
            }
        )


diagnostic_prediction_distribution = pd.DataFrame(
    prediction_distribution_records
)


# ------------------------------------------------------------
# 4. Prediction-confidence summaries
# ------------------------------------------------------------

confidence_records = []

for split_name, split_probabilities in [
    ("Train", logistic_train_probabilities),
    ("Validation", logistic_validation_probabilities),
    ("Test", logistic_test_probabilities),
]:
    maximum_probabilities = (
        split_probabilities.max(axis=1)
    )

    confidence_records.append(
        {
            "Split": split_name,
            "Minimum": maximum_probabilities.min(),
            "25thPercentile": maximum_probabilities.quantile(0.25),
            "Median": maximum_probabilities.median(),
            "Mean": maximum_probabilities.mean(),
            "75thPercentile": maximum_probabilities.quantile(0.75),
            "Maximum": maximum_probabilities.max(),
            "Above70Percent": (
                maximum_probabilities >= 0.70
            ).mean(),
            "Above80Percent": (
                maximum_probabilities >= 0.80
            ).mean(),
            "Above90Percent": (
                maximum_probabilities >= 0.90
            ).mean(),
        }
    )


diagnostic_confidence = pd.DataFrame(
    confidence_records
)

probability_columns_to_round = [
    column
    for column in diagnostic_confidence.columns
    if column != "Split"
]

diagnostic_confidence[
    probability_columns_to_round
] = diagnostic_confidence[
    probability_columns_to_round
].round(6)


# ------------------------------------------------------------
# 5. Confusion matrices
# ------------------------------------------------------------

training_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        y_train,
        logistic_train_predictions,
        labels=class_order,
    ),
    index=[
        f"Actual_{outcome}"
        for outcome in class_order
    ],
    columns=[
        f"Predicted_{outcome}"
        for outcome in class_order
    ],
)

validation_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        y_validation,
        logistic_validation_predictions,
        labels=class_order,
    ),
    index=[
        f"Actual_{outcome}"
        for outcome in class_order
    ],
    columns=[
        f"Predicted_{outcome}"
        for outcome in class_order
    ],
)

test_confusion_matrix = pd.DataFrame(
    confusion_matrix(
        y_test,
        logistic_test_predictions,
        labels=class_order,
    ),
    index=[
        f"Actual_{outcome}"
        for outcome in class_order
    ],
    columns=[
        f"Predicted_{outcome}"
        for outcome in class_order
    ],
)


# ------------------------------------------------------------
# 6. Find highest-confidence incorrect predictions
# ------------------------------------------------------------

def create_error_diagnostic_table(
    metadata,
    actual_outcomes,
    predicted_outcomes,
    probabilities,
    number_of_rows=15,
):
    """
    Return the most confident incorrect predictions.
    """
    error_table = metadata.copy()

    error_table["ActualOutcome"] = actual_outcomes
    error_table["PredictedOutcome"] = predicted_outcomes

    for outcome in class_order:
        error_table[
            f"Probability_{outcome}"
        ] = probabilities[outcome]

    error_table["MaximumProbability"] = (
        probabilities.max(axis=1)
    )

    actual_probability = np.array(
        [
            probabilities.loc[
                row_index,
                actual_outcome,
            ]
            for row_index, actual_outcome
            in actual_outcomes.items()
        ]
    )

    error_table["ProbabilityOfActualOutcome"] = (
        actual_probability
    )

    error_table["FixtureLogLoss"] = (
        -np.log(
            np.clip(
                error_table[
                    "ProbabilityOfActualOutcome"
                ],
                1e-15,
                1.0,
            )
        )
    )

    error_table = (
        error_table[
            error_table["ActualOutcome"]
            != error_table["PredictedOutcome"]
        ]
        .sort_values(
            by=[
                "MaximumProbability",
                "FixtureLogLoss",
            ],
            ascending=False,
            kind="mergesort",
        )
        .head(number_of_rows)
        .reset_index(drop=True)
    )

    return error_table


worst_validation_predictions = (
    create_error_diagnostic_table(
        metadata=metadata_validation,
        actual_outcomes=y_validation,
        predicted_outcomes=logistic_validation_predictions,
        probabilities=logistic_validation_probabilities,
    )
)

worst_test_predictions = (
    create_error_diagnostic_table(
        metadata=metadata_test,
        actual_outcomes=y_test,
        predicted_outcomes=logistic_test_predictions,
        probabilities=logistic_test_probabilities,
    )
)


# ------------------------------------------------------------
# 7. Largest model coefficients
# ------------------------------------------------------------

diagnostic_largest_coefficients = (
    logistic_coefficients
    .sort_values(
        by=[
            "Outcome",
            "AbsoluteCoefficient",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .groupby(
        "Outcome",
        sort=False,
    )
    .head(10)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Probability and alignment sanity checks
# ------------------------------------------------------------

alignment_checks = pd.DataFrame(
    {
        "Check": [
            "Stored model class order",
            "Required probability order",
            "Training indices aligned",
            "Validation indices aligned",
            "Test indices aligned",
            "Training rows sum to one",
            "Validation rows sum to one",
            "Test rows sum to one",
            "Model iterations",
            "Maximum iterations allowed",
            "Converged before maximum",
        ],
        "Result": [
            str(list(logistic_model.classes_)),
            str(class_order),
            logistic_train_probabilities.index.equals(
                y_train.index
            ),
            logistic_validation_probabilities.index.equals(
                y_validation.index
            ),
            logistic_test_probabilities.index.equals(
                y_test.index
            ),
            np.allclose(
                logistic_train_probabilities.sum(axis=1),
                1.0,
            ),
            np.allclose(
                logistic_validation_probabilities.sum(axis=1),
                1.0,
            ),
            np.allclose(
                logistic_test_probabilities.sum(axis=1),
                1.0,
            ),
            int(logistic_model.n_iter_.max()),
            int(logistic_model.max_iter),
            bool(
                logistic_model.n_iter_.max()
                < logistic_model.max_iter
            ),
        ],
    }
)


# ------------------------------------------------------------
# 9. Display the full diagnostic report
# ------------------------------------------------------------

print("=" * 70)
print("LOGISTIC REGRESSION DIAGNOSTIC REPORT")
print("=" * 70)

print("\n1. PERFORMANCE BY SPLIT")
display(diagnostic_performance)

print("\n2. IMPROVEMENT OVER NAIVE BENCHMARKS")
display(logistic_improvement_summary)

print("\n3. PREDICTED-CLASS DISTRIBUTION")
display(diagnostic_prediction_distribution)

print("\n4. PREDICTION CONFIDENCE")
display(diagnostic_confidence)

print("\n5. TRAINING CONFUSION MATRIX")
display(training_confusion_matrix)

print("\n6. VALIDATION CONFUSION MATRIX")
display(validation_confusion_matrix)

print("\n7. TEST CONFUSION MATRIX")
display(test_confusion_matrix)

print("\n8. MOST CONFIDENT INCORRECT VALIDATION PREDICTIONS")
display(worst_validation_predictions)

print("\n9. MOST CONFIDENT INCORRECT TEST PREDICTIONS")
display(worst_test_predictions)

print("\n10. TEN LARGEST COEFFICIENTS PER OUTCOME")
display(diagnostic_largest_coefficients)

print("\n11. CLASS-ORDER, ALIGNMENT AND CONVERGENCE CHECKS")
display(alignment_checks)

LOGISTIC REGRESSION DIAGNOSTIC REPORT

1. PERFORMANCE BY SPLIT


,Split,Fixtures,LogLoss,BrierScore,Accuracy
0,Train,3040,0.957654,0.568314,0.542105
1,Validation,380,0.941668,0.551312,0.589474
2,Test,380,1.012463,0.604957,0.505263



2. IMPROVEMENT OVER NAIVE BENCHMARKS


,Split,BestNaiveLogLoss,LogisticLogLoss,LogLossImprovement,BestNaiveBrierScore,LogisticBrierScore,BrierImprovement
0,Validation,1.054044,0.941668,0.112376,0.637099,0.551312,0.085787
1,Test,1.080909,1.012463,0.068446,0.655601,0.604957,0.050644



3. PREDICTED-CLASS DISTRIBUTION


...

,Season,Date,HomeTeam,AwayTeam,ActualOutcome,PredictedOutcome,Probability_H,Probability_D,Probability_A,MaximumProbability,ProbabilityOfActualOutcome,FixtureLogLoss
0,2024-25,2025-04-23,Arsenal,Crystal Palace,D,H,0.905302,0.051253,0.043445,0.905302,0.051253,2.970987
1,2024-25,2025-04-12,Arsenal,Brentford,D,H,0.833560,0.090632,0.075807,0.833560,0.090632,2.400946
2,2024-25,2025-02-22,Arsenal,West Ham,A,H,0.824634,0.121361,0.054004,0.824634,0.054004,2.918693
3,2024-25,2024-12-14,Arsenal,Everton,D,H,0.819307,0.121252,0.059441,0.819307,0.121252,2.109884
4,2024-25,2025-05-10,Southampton,Man City,D,A,0.062734,0.121306,0.815960,0.815960,0.121306,2.109442
5,2024-25,2024-09-14,Liverpool,Nott'm Forest,A,H,0.805021,0.125984,0.068995,0.805021,0.068995,2.673724
6,2024-25,2024-12-22,Fulham,Southampton,D,H,0.803197,0.120460,0.076342,0.803197,0.120460,2.116434
7,2024-25,2025-05-03,Arsenal,Bournemouth,A,H,0.800580,0.112071,0.087349,0.800580,0.087349,2.437842
8,2024-25,2025-01-05,Liverpool,Man United,D,H,0.793709,0.143011,0.063280,0.793709,0.143011,1.944835
9,2024-25,2024-12-07,Crystal Palace,Man City,D,A,0.114884,0.100695,0.784421,0.784421,0.100695,2.295664



10. TEN LARGEST COEFFICIENTS PER OUTCOME


,Outcome,Feature,Coefficient,AbsoluteCoefficient
0,A,HomeRollingGoalsFor5,-0.383286,0.383286
1,A,VenuePointsFormDifference5,-0.297321,0.297321
2,A,GoalsForFormDifference5,0.293799,0.293799
3,A,EloDifference,-0.242144,0.242144
4,A,AwayRollingGoalsFor5,0.228340,0.228340
5,A,GoalDifferenceFormDifference5,0.208426,0.208426
6,A,AwayRollingGoalDifference5,0.189568,0.189568
7,A,VenueGoalsForFormDifference5,0.188907,0.188907
8,A,HomeRollingGoalsAgainst5,0.186385,0.186385
9,A,HomeEloBefore,-0.177306,0.177306



11. CLASS-ORDER, ALIGNMENT AND CONVERGENCE CHECKS


,Check,Result
0,Stored model class order,"['A', 'D', 'H']"
1,Required probability order,"['H', 'D', 'A']"
2,Training indices aligned,True
3,Validation indices aligned,True
4,Test indices aligned,True
5,Training rows sum to one,True
6,Validation rows sum to one,True
7,Test rows sum to one,True
8,Model iterations,156
9,Maximum iterations allowed,5000


### Diagnostic Interpretation

The diagnostic report verifies the mechanics of the initial multinomial logistic-regression baseline before any hyperparameter tuning is performed.

The checks confirm that:

- predictor and target indices remain aligned;
- probability rows correspond to the correct fixtures;
- model probabilities are explicitly displayed in the project order `(H, D, A)`;
- each probability row sums to one;
- the optimiser converges before reaching its iteration limit;
- predicted probabilities vary across fixtures rather than remaining constant.

Log loss is class-order sensitive. The notebook therefore evaluates it with the custom `multiclass_log_loss` helper, which directly selects the probability assigned to each observed result and independently verifies the answer against scikit-learn using its required lexicographic class order.

The initial model should be interpreted as an untuned linear benchmark. Its validation log loss and Brier score should be compared directly with the uniform and historical-frequency forecasts shown above. Draw prediction rates, confusion matrices and high-confidence errors provide secondary diagnostic evidence, but they do not replace probability-based evaluation.

The next section tests whether regularisation strength and class weighting improve validation probability quality. Hyperparameters will be selected using validation log loss only, while the test result will not influence the selection decision.

## 9. Logistic Regression Hyperparameter Optimisation

The initial multinomial logistic-regression model was fitted using the default regularisation setting:

$$
C=1
$$

with no class weighting.

This section tests whether alternative regularisation strengths and class-weighting choices improve out-of-sample probability quality.

### Regularisation Strength

The logistic-regression parameter $C$ is the inverse of regularisation strength:

$$
C=\frac{1}{\lambda}.
$$

Therefore:

- smaller values of $C$ apply stronger regularisation;
- larger values of $C$ apply weaker regularisation.

Stronger regularisation shrinks coefficients towards zero and may improve generalisation when predictors are correlated or when the default model produces probabilities that are too extreme.

The following candidate values will be tested:

$$
C \in
\{
0.001,\ 0.01,\ 0.1,\ 1,\ 10,\ 100
\}.
$$

### Class Weighting

The original model treats every training fixture equally. Because home wins, draws and away wins do not occur with equal frequency, a balanced class-weighting specification will also be evaluated.

With balanced class weights, the weight assigned to class $k$ is approximately:

$$
w_k
=
\frac{N}{K N_k},
$$

where $N$ is the number of training fixtures, $K$ is the number of outcome classes and $N_k$ is the number of training fixtures in class $k$.

Balanced weighting may improve the model's treatment of less frequent outcomes, particularly draws, but it can also distort probability calibration. It must therefore be judged using validation log loss rather than draw accuracy alone.

### Hyperparameter Grid

The search evaluates every combination of:

$$
C \in
\{
0.001,\ 0.01,\ 0.1,\ 1,\ 10,\ 100
\}
$$

and:

$$
\text{class weight}
\in
\{
\text{None},\ \text{balanced}
\}.
$$

This produces $6 \times 2 = 12$ candidate models.

Every candidate is trained using only `X_train_processed` and `y_train`. The primary selection criterion is validation log loss:

$$
(C^*,w^*)
=
\arg\min_{C,w}
\operatorname{LogLoss}_{\text{validation}}(C,w).
$$

Validation Brier score, accuracy, predicted draw frequency, average confidence and convergence information are retained as diagnostics, but they do not override the primary selection criterion.

If several candidates are effectively tied within the stated numerical tolerance, the more strongly regularised model is preferred.

After selection, the locked configuration is evaluated on the test season. The test result is reported as a final assessment and does not participate in hyperparameter selection.

In [9]:
# ============================================================
# 9. Logistic Regression Hyperparameter Optimisation
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# ------------------------------------------------------------
# Define the hyperparameter search grid
# ------------------------------------------------------------

candidate_C_values = [
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
]

candidate_class_weights = [
    ("None", None),
    ("Balanced", "balanced"),
]

expected_candidate_count = (
    len(candidate_C_values)
    * len(candidate_class_weights)
)


# ------------------------------------------------------------
# Fit every candidate using training data only
# ------------------------------------------------------------

logistic_tuning_records = []
fitted_logistic_candidates = {}

for C_value in candidate_C_values:

    for class_weight_label, class_weight_value in (
        candidate_class_weights
    ):

        candidate_model = LogisticRegression(
            C=C_value,
            class_weight=class_weight_value,
            solver="lbfgs",
            max_iter=5000,
        )

        candidate_model.fit(
            X_train_processed,
            y_train,
        )

        candidate_key = (
            float(C_value),
            class_weight_label,
        )

        fitted_logistic_candidates[
            candidate_key
        ] = candidate_model


        # ----------------------------------------------------
        # Generate validation probabilities
        # ----------------------------------------------------

        candidate_validation_probabilities = (
            create_aligned_probability_frame(
                fitted_model=candidate_model,
                processed_features=X_validation_processed,
                desired_class_order=class_order,
            )
        )

        validate_probability_forecast(
            probability_frame=(
                candidate_validation_probabilities
            ),
            expected_index=y_validation.index,
            labels=class_order,
            name=(
                f"C={C_value}, "
                f"class_weight={class_weight_label}"
            ),
        )


        # ----------------------------------------------------
        # Generate validation class predictions
        # ----------------------------------------------------

        candidate_validation_predictions = (
            candidate_validation_probabilities
            .idxmax(axis=1)
        )

        draw_prediction_rate = (
            candidate_validation_predictions
            .eq("D")
            .mean()
        )

        average_draw_probability = (
            candidate_validation_probabilities[
                "D"
            ]
            .mean()
        )

        average_maximum_probability = (
            candidate_validation_probabilities
            .max(axis=1)
            .mean()
        )


        # ----------------------------------------------------
        # Record candidate performance
        # ----------------------------------------------------

        logistic_tuning_records.append(
            {
                "C": float(C_value),
                "ClassWeight": class_weight_label,
                "ValidationLogLoss": multiclass_log_loss(
                    y_validation,
                    candidate_validation_probabilities,
                    labels=class_order,
                ),
                "ValidationBrierScore": (
                    multiclass_brier_score(
                        y_validation,
                        candidate_validation_probabilities,
                        labels=class_order,
                    )
                ),
                "ValidationAccuracy": accuracy_score(
                    y_validation,
                    candidate_validation_predictions,
                ),
                "PredictedDrawRate": (
                    draw_prediction_rate
                ),
                "AverageDrawProbability": (
                    average_draw_probability
                ),
                "AverageMaximumProbability": (
                    average_maximum_probability
                ),
                "Iterations": int(
                    candidate_model.n_iter_.max()
                ),
                "Converged": bool(
                    candidate_model.n_iter_.max()
                    < candidate_model.max_iter
                ),
            }
        )


logistic_tuning_results = pd.DataFrame(
    logistic_tuning_records
)


# ------------------------------------------------------------
# Validate the completed search
# ------------------------------------------------------------

assert len(logistic_tuning_results) == (
    expected_candidate_count
), (
    "The tuning table does not contain every expected "
    "hyperparameter combination."
)

assert logistic_tuning_results[
    "Converged"
].all(), (
    "At least one logistic-regression candidate did not converge."
)

assert logistic_tuning_results[
    "ValidationLogLoss"
].ge(0).all(), (
    "Validation log loss cannot be negative."
)

assert logistic_tuning_results[
    "ValidationBrierScore"
].ge(0).all(), (
    "Validation Brier score cannot be negative."
)

assert logistic_tuning_results[
    "ValidationAccuracy"
].between(0, 1).all(), (
    "Validation accuracy must remain between zero and one."
)

assert logistic_tuning_results[
    "PredictedDrawRate"
].between(0, 1).all(), (
    "Predicted draw rate must remain between zero and one."
)

assert logistic_tuning_results[
    "AverageDrawProbability"
].between(0, 1).all(), (
    "Average draw probability must remain between zero and one."
)

assert logistic_tuning_results[
    "AverageMaximumProbability"
].between(0, 1).all(), (
    "Average maximum probability must remain between zero and one."
)


# ------------------------------------------------------------
# Select the best candidate using validation log loss only
# ------------------------------------------------------------

best_validation_log_loss = (
    logistic_tuning_results[
        "ValidationLogLoss"
    ].min()
)

selection_tolerance = 1e-6

near_best_candidates = (
    logistic_tuning_results[
        logistic_tuning_results[
            "ValidationLogLoss"
        ]
        <= (
            best_validation_log_loss
            + selection_tolerance
        )
    ]
    .copy()
)


# Prefer no class weighting when every other criterion is tied.
near_best_candidates[
    "ClassWeightPriority"
] = (
    near_best_candidates[
        "ClassWeight"
    ]
    .map(
        {
            "None": 0,
            "Balanced": 1,
        }
    )
)


# Smaller C means stronger regularisation.
best_candidate_row = (
    near_best_candidates
    .sort_values(
        by=[
            "C",
            "ClassWeightPriority",
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .iloc[0]
)


best_logistic_C = float(
    best_candidate_row["C"]
)

best_logistic_class_weight_label = (
    best_candidate_row["ClassWeight"]
)

best_logistic_class_weight = (
    None
    if best_logistic_class_weight_label == "None"
    else "balanced"
)


# ------------------------------------------------------------
# Retrieve the already fitted winning model
# ------------------------------------------------------------

best_candidate_key = (
    best_logistic_C,
    best_logistic_class_weight_label,
)

tuned_logistic_model = (
    fitted_logistic_candidates[
        best_candidate_key
    ]
)


# ------------------------------------------------------------
# Generate locked validation and test probabilities
# ------------------------------------------------------------

tuned_logistic_validation_probabilities = (
    create_aligned_probability_frame(
        fitted_model=tuned_logistic_model,
        processed_features=X_validation_processed,
        desired_class_order=class_order,
    )
)

tuned_logistic_test_probabilities = (
    create_aligned_probability_frame(
        fitted_model=tuned_logistic_model,
        processed_features=X_test_processed,
        desired_class_order=class_order,
    )
)


validate_probability_forecast(
    probability_frame=(
        tuned_logistic_validation_probabilities
    ),
    expected_index=y_validation.index,
    labels=class_order,
    name="tuned logistic validation probabilities",
)

validate_probability_forecast(
    probability_frame=(
        tuned_logistic_test_probabilities
    ),
    expected_index=y_test.index,
    labels=class_order,
    name="tuned logistic test probabilities",
)


# ------------------------------------------------------------
# Create tuned class predictions
# ------------------------------------------------------------

tuned_logistic_validation_predictions = (
    tuned_logistic_validation_probabilities
    .idxmax(axis=1)
    .rename("PredictedOutcome")
)

tuned_logistic_test_predictions = (
    tuned_logistic_test_probabilities
    .idxmax(axis=1)
    .rename("PredictedOutcome")
)


# ------------------------------------------------------------
# Evaluate the locked tuned model
# ------------------------------------------------------------

tuned_logistic_results = pd.DataFrame(
    [
        {
            "Model": "Tuned Logistic Regression",
            "Split": "Validation",
            "Fixtures": len(y_validation),
            "LogLoss": multiclass_log_loss(
                y_validation,
                tuned_logistic_validation_probabilities,
                labels=class_order,
            ),
            "BrierScore": multiclass_brier_score(
                y_validation,
                tuned_logistic_validation_probabilities,
                labels=class_order,
            ),
            "Accuracy": accuracy_score(
                y_validation,
                tuned_logistic_validation_predictions,
            ),
        },
        {
            "Model": "Tuned Logistic Regression",
            "Split": "Test",
            "Fixtures": len(y_test),
            "LogLoss": multiclass_log_loss(
                y_test,
                tuned_logistic_test_probabilities,
                labels=class_order,
            ),
            "BrierScore": multiclass_brier_score(
                y_test,
                tuned_logistic_test_probabilities,
                labels=class_order,
            ),
            "Accuracy": accuracy_score(
                y_test,
                tuned_logistic_test_predictions,
            ),
        },
    ]
)


# ------------------------------------------------------------
# Compare with the untuned and naive models
# ------------------------------------------------------------

naive_results_for_comparison = (
    benchmark_results[
        [
            "Benchmark",
            "Split",
            "Fixtures",
            "LogLoss",
            "BrierScore",
            "Accuracy",
        ]
    ]
    .rename(
        columns={
            "Benchmark": "Model",
        }
    )
    .copy()
)

untuned_logistic_for_comparison = (
    logistic_results[
        [
            "Model",
            "Split",
            "Fixtures",
            "LogLoss",
            "BrierScore",
            "Accuracy",
        ]
    ]
    .copy()
)

tuned_logistic_for_comparison = (
    tuned_logistic_results[
        [
            "Model",
            "Split",
            "Fixtures",
            "LogLoss",
            "BrierScore",
            "Accuracy",
        ]
    ]
    .copy()
)

tuned_model_comparison = pd.concat(
    [
        naive_results_for_comparison,
        untuned_logistic_for_comparison,
        tuned_logistic_for_comparison,
    ],
    ignore_index=True,
)

for metric_column in [
    "LogLoss",
    "BrierScore",
    "Accuracy",
]:
    tuned_model_comparison[
        metric_column
    ] = (
        tuned_model_comparison[
            metric_column
        ]
        .astype(float)
        .round(6)
    )

tuned_model_comparison = (
    tuned_model_comparison
    .sort_values(
        by=[
            "Split",
            "LogLoss",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Measure improvement over the untuned model
# ------------------------------------------------------------

tuned_improvement_records = []

for split_name in [
    "Validation",
    "Test",
]:

    untuned_result = (
        logistic_results[
            logistic_results["Split"]
            == split_name
        ]
        .iloc[0]
    )

    tuned_result = (
        tuned_logistic_results[
            tuned_logistic_results["Split"]
            == split_name
        ]
        .iloc[0]
    )

    tuned_improvement_records.append(
        {
            "Split": split_name,
            "UntunedLogLoss": (
                untuned_result["LogLoss"]
            ),
            "TunedLogLoss": (
                tuned_result["LogLoss"]
            ),
            "LogLossImprovement": (
                untuned_result["LogLoss"]
                - tuned_result["LogLoss"]
            ),
            "UntunedBrierScore": (
                untuned_result["BrierScore"]
            ),
            "TunedBrierScore": (
                tuned_result["BrierScore"]
            ),
            "BrierImprovement": (
                untuned_result["BrierScore"]
                - tuned_result["BrierScore"]
            ),
            "UntunedAccuracy": (
                untuned_result["Accuracy"]
            ),
            "TunedAccuracy": (
                tuned_result["Accuracy"]
            ),
        }
    )


tuned_logistic_improvement = pd.DataFrame(
    tuned_improvement_records
)

for column in [
    "UntunedLogLoss",
    "TunedLogLoss",
    "LogLossImprovement",
    "UntunedBrierScore",
    "TunedBrierScore",
    "BrierImprovement",
    "UntunedAccuracy",
    "TunedAccuracy",
]:
    tuned_logistic_improvement[
        column
    ] = (
        tuned_logistic_improvement[
            column
        ]
        .astype(float)
        .round(6)
    )


# ------------------------------------------------------------
# Summarise tuned predicted-class behaviour
# ------------------------------------------------------------

tuned_prediction_distribution = pd.concat(
    [
        (
            tuned_logistic_validation_predictions
            .value_counts()
            .reindex(
                class_order,
                fill_value=0,
            )
            .rename_axis("Outcome")
            .reset_index(
                name="PredictedFixtures"
            )
            .assign(Split="Validation")
        ),
        (
            tuned_logistic_test_predictions
            .value_counts()
            .reindex(
                class_order,
                fill_value=0,
            )
            .rename_axis("Outcome")
            .reset_index(
                name="PredictedFixtures"
            )
            .assign(Split="Test")
        ),
    ],
    ignore_index=True,
)

tuned_prediction_distribution[
    "PredictedPercentage"
] = (
    tuned_prediction_distribution
    .groupby("Split")[
        "PredictedFixtures"
    ]
    .transform(
        lambda counts: (
            100
            * counts
            / counts.sum()
        ).round(2)
    )
)


# ------------------------------------------------------------
# Create fixture-level tuned prediction tables
# ------------------------------------------------------------

tuned_logistic_validation_output = (
    metadata_validation.copy()
)

tuned_logistic_validation_output[
    "ActualOutcome"
] = y_validation

tuned_logistic_validation_output[
    "PredictedOutcome"
] = (
    tuned_logistic_validation_predictions
)

for outcome in class_order:
    tuned_logistic_validation_output[
        f"Probability_{outcome}"
    ] = (
        tuned_logistic_validation_probabilities[
            outcome
        ]
    )


tuned_logistic_test_output = (
    metadata_test.copy()
)

tuned_logistic_test_output[
    "ActualOutcome"
] = y_test

tuned_logistic_test_output[
    "PredictedOutcome"
] = (
    tuned_logistic_test_predictions
)

for outcome in class_order:
    tuned_logistic_test_output[
        f"Probability_{outcome}"
    ] = (
        tuned_logistic_test_probabilities[
            outcome
        ]
    )


# ------------------------------------------------------------
# Format tuning results for display
# ------------------------------------------------------------

display_tuning_results = (
    logistic_tuning_results
    .copy()
)

for column in [
    "ValidationLogLoss",
    "ValidationBrierScore",
    "ValidationAccuracy",
    "PredictedDrawRate",
    "AverageDrawProbability",
    "AverageMaximumProbability",
]:
    display_tuning_results[column] = (
        display_tuning_results[column]
        .astype(float)
        .round(6)
    )

display_tuning_results = (
    display_tuning_results
    .sort_values(
        by=[
            "ValidationLogLoss",
            "C",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert (
    tuned_logistic_model.n_iter_.max()
    < tuned_logistic_model.max_iter
), "The selected tuned model did not converge."

assert best_logistic_C in candidate_C_values, (
    "The selected C value was not in the search grid."
)

assert (
    best_logistic_class_weight_label
    in {"None", "Balanced"}
), "An invalid class-weight setting was selected."

assert np.isclose(
    tuned_logistic_results.loc[
        tuned_logistic_results["Split"]
        == "Validation",
        "LogLoss",
    ].iloc[0],
    best_validation_log_loss,
), (
    "The selected tuned model does not match the "
    "lowest validation log loss."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "Logistic-regression hyperparameter search "
    "completed successfully."
)

print(f"Best C: {best_logistic_C}")

print(
    "Best class weight:",
    best_logistic_class_weight_label,
)

print(
    "Best validation log loss:",
    round(
        best_validation_log_loss,
        6,
    ),
)

display(display_tuning_results)

display(tuned_model_comparison)

display(tuned_logistic_improvement)

display(tuned_prediction_distribution)

display(
    tuned_logistic_validation_output.head(20)
)

Logistic-regression hyperparameter search completed successfully.
Best C: 0.01
Best class weight: None
Best validation log loss: 0.934551


,C,ClassWeight,ValidationLogLoss,ValidationBrierScore,ValidationAccuracy,PredictedDrawRate,AverageDrawProbability,AverageMaximumProbability,Iterations,Converged
0,0.010,None,0.934551,0.549322,0.592105,0.007895,0.220865,0.562329,30,True
1,0.100,None,0.937148,0.549510,0.586842,0.010526,0.218442,0.571905,70,True
2,1.000,None,0.941668,0.551312,0.589474,0.015789,0.218610,0.574196,156,True
3,0.001,None,0.943684,0.557002,0.592105,0.000000,0.227156,0.532327,16,True
4,10.000,None,0.957946,0.558853,0.586842,0.021053,0.220717,0.576874,320,True
5,0.010,Balanced,0.965267,0.572582,0.536842,0.160526,0.315066,0.500814,32,True
6,0.100,Balanced,0.965666,0.571760,0.536842,0.178947,0.312422,0.513027,71,True
7,1.000,Balanced,0.971498,0.574002,0.536842,0.178947,0.312426,0.516219,186,True
8,0.001,Balanced,0.980485,0.583887,0.518421,0.173684,0.323397,0.465858,15,True
9,10.000,Balanced,0.990229,0.582067,0.528947,0.192105,0.314146,0.520063,312,True


,Model,Split,Fixtures,LogLoss,BrierScore,Accuracy
0,Tuned Logistic Regression,Test,380,1.004062,0.601040,0.505263
1,Multinomial Logistic Regression,Test,380,1.012463,0.604957,0.505263
2,Historical Frequency,Test,380,1.080909,0.655601,0.407895
3,Uniform Probability,Test,380,1.098612,0.666667,0.407895
4,Tuned Logistic Regression,Validation,380,0.934551,0.549322,0.592105
5,Multinomial Logistic Regression,Validation,380,0.941668,0.551312,0.589474
6,Historical Frequency,Validation,380,1.054044,0.637099,0.460526
7,Uniform Probability,Validation,380,1.098612,0.666667,0.460526


,Split,UntunedLogLoss,TunedLogLoss,LogLossImprovement,UntunedBrierScore,TunedBrierScore,BrierImprovement,UntunedAccuracy,TunedAccuracy
0,Validation,0.941668,0.934551,0.007117,0.551312,0.549322,0.001990,0.589474,0.592105
1,Test,1.012463,1.004062,0.008401,0.604957,0.601040,0.003917,0.505263,0.505263


,Outcome,PredictedFixtures,Split,PredictedPercentage
0,H,245,Validation,64.47
1,D,3,Validation,0.79
2,A,132,Validation,34.74
3,H,246,Test,64.74
4,D,2,Test,0.53
5,A,132,Test,34.74


,Season,Date,HomeTeam,AwayTeam,ActualOutcome,PredictedOutcome,Probability_H,Probability_D,Probability_A
3040,2023-24,2023-08-11,Burnley,Man City,A,A,0.136984,0.174210,0.688806
3041,2023-24,2023-08-12,Arsenal,Nott'm Forest,H,H,0.725843,0.154762,0.119396
3042,2023-24,2023-08-12,Bournemouth,West Ham,D,H,0.426104,0.219167,0.354728
3043,2023-24,2023-08-12,Brighton,Luton,H,H,0.538364,0.240616,0.221020
3044,2023-24,2023-08-12,Everton,Fulham,A,H,0.351596,0.302309,0.346095
3045,2023-24,2023-08-12,Newcastle,Aston Villa,H,H,0.444235,0.257884,0.297880
3046,2023-24,2023-08-12,Sheffield United,Crystal Palace,A,A,0.253325,0.287983,0.458691
3047,2023-24,2023-08-13,Brentford,Tottenham,D,H,0.414567,0.241744,0.343689
3048,2023-24,2023-08-13,Chelsea,Liverpool,D,A,0.265296,0.206844,0.527860
3049,2023-24,2023-08-14,Man United,Wolves,H,H,0.598658,0.229049,0.172293


In [10]:
# ============================================================
# Final Probability-Alignment Verification
# ============================================================

# 1. Check target encoding
print("Observed target values:")
print(sorted(y_train.unique()))

print("\nTarget counts:")
print(y_train.value_counts().reindex(class_order))


# 2. Check required project class order
print("\nRequired class order:")
print(class_order)


# 3. Check model's internal class order
print("\nModel internal class order:")
print(list(tuned_logistic_model.classes_))


# 4. Generate raw, unaligned probabilities
raw_validation_probabilities = tuned_logistic_model.predict_proba(
    X_validation_processed
)

raw_validation_probability_frame = pd.DataFrame(
    raw_validation_probabilities,
    index=X_validation_processed.index,
    columns=tuned_logistic_model.classes_,
)

print("\nRaw probability columns:")
print(list(raw_validation_probability_frame.columns))


# 5. Reorder probabilities explicitly
reordered_validation_probability_frame = (
    raw_validation_probability_frame[class_order]
    .copy()
)

print("\nReordered probability columns:")
print(list(reordered_validation_probability_frame.columns))


# 6. Confirm the stored tuned probabilities are identical
assert np.allclose(
    reordered_validation_probability_frame.to_numpy(),
    tuned_logistic_validation_probabilities.to_numpy(),
), (
    "Stored tuned validation probabilities do not match "
    "the explicitly reordered model probabilities."
)


# 7. Confirm rows and targets align
assert reordered_validation_probability_frame.index.equals(
    y_validation.index
), "Validation probabilities and targets are not index-aligned."


# 8. Recalculate metrics with the class-order-safe helper
independent_validation_log_loss = multiclass_log_loss(
    y_validation,
    reordered_validation_probability_frame,
    labels=class_order,
)

independent_validation_brier = multiclass_brier_score(
    y_validation,
    reordered_validation_probability_frame,
    class_order,
)

print("\nIndependent validation metrics:")
print(
    "Log loss:",
    round(independent_validation_log_loss, 6),
)
print(
    "Brier score:",
    round(independent_validation_brier, 6),
)


# 9. Manual row-level check
manual_check = metadata_validation.copy()

manual_check["ActualOutcome"] = y_validation

for outcome in class_order:
    manual_check[f"Probability_{outcome}"] = (
        reordered_validation_probability_frame[outcome]
    )

manual_check["ProbabilityOfActualOutcome"] = [
    reordered_validation_probability_frame.loc[index, outcome]
    for index, outcome in y_validation.items()
]

manual_check["RowLogLossContribution"] = (
    -np.log(
        manual_check["ProbabilityOfActualOutcome"]
    )
)

manual_mean_log_loss = float(
    manual_check["RowLogLossContribution"].mean()
)

assert np.isclose(
    independent_validation_log_loss,
    manual_mean_log_loss,
), (
    "The helper log loss and mean row-level contribution do not agree."
)

print("\nProbability-alignment verification passed.")
print(
    "Mean row log-loss contribution:",
    round(manual_mean_log_loss, 6),
)

display(
    manual_check[
        [
            season_column,
            date_column,
            home_team_column,
            away_team_column,
            "ActualOutcome",
            "Probability_H",
            "Probability_D",
            "Probability_A",
            "ProbabilityOfActualOutcome",
            "RowLogLossContribution",
        ]
    ].head(20)
)

Observed target values:
['A', 'D', 'H']

Target counts:
FTR
H    1361
D     711
A     968
Name: count, dtype: int64[pyarrow]

Required class order:
['H', 'D', 'A']

Model internal class order:
['A', 'D', 'H']

Raw probability columns:
['A', 'D', 'H']

Reordered probability columns:
['H', 'D', 'A']

Independent validation metrics:
Log loss: 0.934551
Brier score: 0.549322

Probability-alignment verification passed.
Mean row log-loss contribution: 0.934551


,Season,Date,HomeTeam,AwayTeam,ActualOutcome,Probability_H,Probability_D,Probability_A,ProbabilityOfActualOutcome,RowLogLossContribution
3040,2023-24,2023-08-11,Burnley,Man City,A,0.136984,0.174210,0.688806,0.688806,0.372796
3041,2023-24,2023-08-12,Arsenal,Nott'm Forest,H,0.725843,0.154762,0.119396,0.725843,0.320422
3042,2023-24,2023-08-12,Bournemouth,West Ham,D,0.426104,0.219167,0.354728,0.219167,1.517919
3043,2023-24,2023-08-12,Brighton,Luton,H,0.538364,0.240616,0.221020,0.538364,0.619220
3044,2023-24,2023-08-12,Everton,Fulham,A,0.351596,0.302309,0.346095,0.346095,1.061042
3045,2023-24,2023-08-12,Newcastle,Aston Villa,H,0.444235,0.257884,0.297880,0.444235,0.811401
3046,2023-24,2023-08-12,Sheffield United,Crystal Palace,A,0.253325,0.287983,0.458691,0.458691,0.779378
3047,2023-24,2023-08-13,Brentford,Tottenham,D,0.414567,0.241744,0.343689,0.241744,1.419876
3048,2023-24,2023-08-13,Chelsea,Liverpool,D,0.265296,0.206844,0.527860,0.206844,1.575791
3049,2023-24,2023-08-14,Man United,Wolves,H,0.598658,0.229049,0.172293,0.598658,0.513065


### Interpretation of Hyperparameter Optimisation

The corrected hyperparameter search selected a regularisation strength of `C = 0.01` with no class weighting.

The selected value of `C` is substantially lower than the default value of `1.0`, indicating that stronger regularisation improves out-of-sample probability quality. Stronger regularisation constrains the magnitude of the fitted coefficients, reducing sensitivity to noise, multicollinearity and potentially unstable relationships within the engineered feature set.

Balanced class weighting was not selected. Although class weighting can increase the frequency with which minority outcomes such as draws are predicted, it does not necessarily improve the quality of the full probability distribution. Because the primary optimisation criterion is multiclass log loss, the unweighted specification produced the better validation probabilities.

The tuned model achieved a validation log loss of `0.934551`, outperforming the uniform-probability benchmark of `1.098612`. This provides evidence that the engineered pre-match features contain meaningful predictive information and that multinomial logistic regression can convert this information into probabilities that are superior to an uninformative forecast.

The result also confirms that the earlier apparent underperformance was caused by an evaluation error rather than a failure of the underlying modelling pipeline. Scikit-learn internally orders the outcome classes lexicographically as `(A, D, H)`, whereas the project reports probabilities in the more intuitive order `(H, D, A)`. The corrected log-loss function explicitly matches each observed outcome to its corresponding probability column, and its result was independently verified using a row-level calculation.

The selected hyperparameters were determined exclusively from validation performance. The configuration is therefore now treated as locked, and its performance on the test season can be used as an independent estimate of generalisation.

The tuned logistic-regression model will be retained as the principal linear baseline against which more flexible nonlinear models are compared in the next modelling stage.

## 10. Final Baseline Evaluation and Model Selection

The hyperparameter search selected a multinomial logistic-regression model with:

$$
C = 0.01
$$

and no class weighting.

This specification was selected using validation log loss only. The independent test season did not influence the hyperparameter-selection decision.

The final stage of this notebook will consolidate the performance of:

- the uniform-probability baseline;
- the historical-frequency baseline;
- the untuned multinomial logistic-regression model;
- the tuned multinomial logistic-regression model.

The models will be compared using:

- multiclass log loss;
- multiclass Brier score;
- classification accuracy.

Log loss remains the primary metric because the eventual purpose of the project is to generate reliable match-outcome probabilities rather than only predict the most likely result.

The final analysis will also:

- quantify the tuned model's improvement over the naive benchmarks;
- compare validation and test performance;
- inspect the largest coefficients from the selected model;
- confirm that the locked model remains correctly aligned with the `(H, D, A)` probability order;
- save the final predictions, performance tables and fitted preprocessing/model objects for use in later notebooks.

The tuned logistic-regression model will serve as the project's official linear baseline. Any subsequent nonlinear model must improve upon its out-of-sample probability performance to justify the additional complexity.

In [11]:
# ============================================================
# 10. Final Baseline Evaluation and Model Selection
# ============================================================

import json

import joblib


# ------------------------------------------------------------
# Create the final comparison table
# ------------------------------------------------------------

final_baseline_comparison = (
    tuned_model_comparison
    .copy()
)

split_order = pd.CategoricalDtype(
    categories=[
        "Validation",
        "Test",
    ],
    ordered=True,
)

final_baseline_comparison["Split"] = (
    final_baseline_comparison["Split"]
    .astype(split_order)
)

final_baseline_comparison["LogLossRank"] = (
    final_baseline_comparison
    .groupby(
        "Split",
        observed=False,
    )["LogLoss"]
    .rank(
        method="dense",
        ascending=True,
    )
    .astype(int)
)

final_baseline_comparison["BrierRank"] = (
    final_baseline_comparison
    .groupby(
        "Split",
        observed=False,
    )["BrierScore"]
    .rank(
        method="dense",
        ascending=True,
    )
    .astype(int)
)

final_baseline_comparison["AccuracyRank"] = (
    final_baseline_comparison
    .groupby(
        "Split",
        observed=False,
    )["Accuracy"]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)

final_baseline_comparison = (
    final_baseline_comparison
    .sort_values(
        by=[
            "Split",
            "LogLossRank",
            "BrierRank",
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Quantify improvement over every simpler benchmark
# ------------------------------------------------------------

baseline_improvement_records = []

reference_model_names = [
    "Uniform Probability",
    "Historical Frequency",
    "Multinomial Logistic Regression",
]

for split_name in [
    "Validation",
    "Test",
]:

    tuned_row = (
        final_baseline_comparison[
            (
                final_baseline_comparison["Split"]
                == split_name
            )
            & (
                final_baseline_comparison["Model"]
                == "Tuned Logistic Regression"
            )
        ]
        .iloc[0]
    )

    for reference_model_name in reference_model_names:

        reference_row = (
            final_baseline_comparison[
                (
                    final_baseline_comparison["Split"]
                    == split_name
                )
                & (
                    final_baseline_comparison["Model"]
                    == reference_model_name
                )
            ]
            .iloc[0]
        )

        log_loss_improvement = (
            float(reference_row["LogLoss"])
            - float(tuned_row["LogLoss"])
        )

        brier_improvement = (
            float(reference_row["BrierScore"])
            - float(tuned_row["BrierScore"])
        )

        accuracy_difference = (
            float(tuned_row["Accuracy"])
            - float(reference_row["Accuracy"])
        )

        baseline_improvement_records.append(
            {
                "Split": split_name,
                "ReferenceModel": reference_model_name,
                "ReferenceLogLoss": float(
                    reference_row["LogLoss"]
                ),
                "TunedLogLoss": float(
                    tuned_row["LogLoss"]
                ),
                "LogLossImprovement": (
                    log_loss_improvement
                ),
                "LogLossImprovementPercentage": (
                    100
                    * log_loss_improvement
                    / float(reference_row["LogLoss"])
                ),
                "ReferenceBrierScore": float(
                    reference_row["BrierScore"]
                ),
                "TunedBrierScore": float(
                    tuned_row["BrierScore"]
                ),
                "BrierImprovement": (
                    brier_improvement
                ),
                "BrierImprovementPercentage": (
                    100
                    * brier_improvement
                    / float(reference_row["BrierScore"])
                ),
                "ReferenceAccuracy": float(
                    reference_row["Accuracy"]
                ),
                "TunedAccuracy": float(
                    tuned_row["Accuracy"]
                ),
                "AccuracyDifferencePercentagePoints": (
                    100 * accuracy_difference
                ),
                "LowerLogLoss": bool(
                    tuned_row["LogLoss"]
                    < reference_row["LogLoss"]
                ),
                "LowerBrierScore": bool(
                    tuned_row["BrierScore"]
                    < reference_row["BrierScore"]
                ),
            }
        )


final_baseline_improvement = pd.DataFrame(
    baseline_improvement_records
)

numeric_improvement_columns = [
    "ReferenceLogLoss",
    "TunedLogLoss",
    "LogLossImprovement",
    "LogLossImprovementPercentage",
    "ReferenceBrierScore",
    "TunedBrierScore",
    "BrierImprovement",
    "BrierImprovementPercentage",
    "ReferenceAccuracy",
    "TunedAccuracy",
    "AccuracyDifferencePercentagePoints",
]

final_baseline_improvement[
    numeric_improvement_columns
] = (
    final_baseline_improvement[
        numeric_improvement_columns
    ]
    .astype(float)
    .round(6)
)


# ------------------------------------------------------------
# Extract the selected model's coefficients
# ------------------------------------------------------------

tuned_coefficient_matrix = pd.DataFrame(
    tuned_logistic_model.coef_,
    index=tuned_logistic_model.classes_,
    columns=processed_feature_names,
)

tuned_coefficient_matrix = (
    tuned_coefficient_matrix
    .loc[class_order]
)

tuned_logistic_coefficients = (
    tuned_coefficient_matrix
    .reset_index(
        names="Outcome",
    )
    .melt(
        id_vars="Outcome",
        var_name="Feature",
        value_name="Coefficient",
    )
)

tuned_logistic_coefficients[
    "AbsoluteCoefficient"
] = (
    tuned_logistic_coefficients[
        "Coefficient"
    ]
    .abs()
)

largest_tuned_coefficients_by_outcome = (
    tuned_logistic_coefficients
    .sort_values(
        by=[
            "Outcome",
            "AbsoluteCoefficient",
        ],
        ascending=[
            True,
            False,
        ],
        kind="mergesort",
    )
    .groupby(
        "Outcome",
        sort=False,
    )
    .head(10)
    .reset_index(drop=True)
)

largest_tuned_coefficients_by_outcome[
    [
        "Coefficient",
        "AbsoluteCoefficient",
    ]
] = (
    largest_tuned_coefficients_by_outcome[
        [
            "Coefficient",
            "AbsoluteCoefficient",
        ]
    ]
    .round(6)
)


# ------------------------------------------------------------
# Add row-level diagnostics to final prediction outputs
# ------------------------------------------------------------

def add_prediction_diagnostics(
    prediction_output,
    probabilities,
):
    """
    Add confidence, correctness and fixture-level log-loss
    information to a prediction output table.
    """
    diagnostic_output = prediction_output.copy()

    actual_probabilities = np.array(
        [
            probabilities.loc[
                row_index,
                actual_outcome,
            ]
            for row_index, actual_outcome
            in diagnostic_output[
                "ActualOutcome"
            ].items()
        ],
        dtype=float,
    )

    diagnostic_output[
        "ProbabilityOfActualOutcome"
    ] = actual_probabilities

    diagnostic_output[
        "MaximumProbability"
    ] = probabilities.max(axis=1)

    diagnostic_output[
        "CorrectPrediction"
    ] = (
        diagnostic_output["ActualOutcome"]
        == diagnostic_output["PredictedOutcome"]
    )

    diagnostic_output[
        "FixtureLogLoss"
    ] = (
        -np.log(
            np.clip(
                actual_probabilities,
                1e-15,
                1.0,
            )
        )
    )

    return diagnostic_output


final_validation_predictions = (
    add_prediction_diagnostics(
        prediction_output=(
            tuned_logistic_validation_output
        ),
        probabilities=(
            tuned_logistic_validation_probabilities
        ),
    )
)

final_test_predictions = (
    add_prediction_diagnostics(
        prediction_output=(
            tuned_logistic_test_output
        ),
        probabilities=(
            tuned_logistic_test_probabilities
        ),
    )
)


# ------------------------------------------------------------
# Create a compact final model summary
# ------------------------------------------------------------

selected_validation_row = (
    tuned_logistic_results[
        tuned_logistic_results["Split"]
        == "Validation"
    ]
    .iloc[0]
)

selected_test_row = (
    tuned_logistic_results[
        tuned_logistic_results["Split"]
        == "Test"
    ]
    .iloc[0]
)

uniform_validation_row = (
    benchmark_results[
        (
            benchmark_results["Benchmark"]
            == "Uniform Probability"
        )
        & (
            benchmark_results["Split"]
            == "Validation"
        )
    ]
    .iloc[0]
)

historical_validation_row = (
    benchmark_results[
        (
            benchmark_results["Benchmark"]
            == "Historical Frequency"
        )
        & (
            benchmark_results["Split"]
            == "Validation"
        )
    ]
    .iloc[0]
)

final_model_summary = pd.DataFrame(
    {
        "Field": [
            "Selected model",
            "Regularisation parameter C",
            "Class weighting",
            "Training seasons",
            "Validation season",
            "Test season",
            "Predictor count",
            "Training fixtures",
            "Validation fixtures",
            "Test fixtures",
            "Validation log loss",
            "Validation Brier score",
            "Validation accuracy",
            "Test log loss",
            "Test Brier score",
            "Test accuracy",
            "Uniform validation log loss",
            "Historical validation log loss",
            "Model probability order",
            "Model internal class order",
        ],
        "Value": [
            "Tuned Multinomial Logistic Regression",
            best_logistic_C,
            best_logistic_class_weight_label,
            ", ".join(
                str(season)
                for season in train_seasons
            ),
            str(validation_season),
            str(test_season),
            len(processed_feature_names),
            len(y_train),
            len(y_validation),
            len(y_test),
            round(
                float(
                    selected_validation_row[
                        "LogLoss"
                    ]
                ),
                6,
            ),
            round(
                float(
                    selected_validation_row[
                        "BrierScore"
                    ]
                ),
                6,
            ),
            round(
                float(
                    selected_validation_row[
                        "Accuracy"
                    ]
                ),
                6,
            ),
            round(
                float(
                    selected_test_row[
                        "LogLoss"
                    ]
                ),
                6,
            ),
            round(
                float(
                    selected_test_row[
                        "BrierScore"
                    ]
                ),
                6,
            ),
            round(
                float(
                    selected_test_row[
                        "Accuracy"
                    ]
                ),
                6,
            ),
            round(
                float(
                    uniform_validation_row[
                        "LogLoss"
                    ]
                ),
                6,
            ),
            round(
                float(
                    historical_validation_row[
                        "LogLoss"
                    ]
                ),
                6,
            ),
            str(class_order),
            str(
                list(
                    tuned_logistic_model.classes_
                )
            ),
        ],
    }
)


# ------------------------------------------------------------
# Final consistency checks
# ------------------------------------------------------------

assert list(
    tuned_logistic_validation_probabilities.columns
) == class_order, (
    "Validation probability columns are not in H, D, A order."
)

assert list(
    tuned_logistic_test_probabilities.columns
) == class_order, (
    "Test probability columns are not in H, D, A order."
)

assert tuned_logistic_validation_probabilities.index.equals(
    y_validation.index
), (
    "Validation probabilities and targets are not aligned."
)

assert tuned_logistic_test_probabilities.index.equals(
    y_test.index
), (
    "Test probabilities and targets are not aligned."
)

assert np.allclose(
    tuned_logistic_validation_probabilities.sum(axis=1),
    1.0,
), "Validation probability rows do not sum to one."

assert np.allclose(
    tuned_logistic_test_probabilities.sum(axis=1),
    1.0,
), "Test probability rows do not sum to one."

assert set(
    tuned_logistic_model.classes_
) == set(class_order), (
    "The selected model does not contain the required classes."
)

assert np.isclose(
    float(
        selected_validation_row[
            "LogLoss"
        ]
    ),
    best_validation_log_loss,
), (
    "The final model does not match the configuration "
    "selected by validation log loss."
)

assert np.isclose(
    final_validation_predictions[
        "FixtureLogLoss"
    ].mean(),
    float(
        selected_validation_row[
            "LogLoss"
        ]
    ),
), (
    "Validation fixture-level log loss does not match "
    "the reported aggregate value."
)

assert np.isclose(
    final_test_predictions[
        "FixtureLogLoss"
    ].mean(),
    float(
        selected_test_row[
            "LogLoss"
        ]
    ),
), (
    "Test fixture-level log loss does not match "
    "the reported aggregate value."
)


# ------------------------------------------------------------
# Save final baseline artefacts
# ------------------------------------------------------------

model_output_directory = (
    project_root
    / "data"
    / "model_outputs"
)

saved_model_directory = (
    project_root
    / "models"
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

saved_model_directory.mkdir(
    parents=True,
    exist_ok=True,
)


comparison_output_path = (
    model_output_directory
    / "baseline_model_comparison.csv"
)

improvement_output_path = (
    model_output_directory
    / "baseline_model_improvement.csv"
)

validation_predictions_path = (
    model_output_directory
    / "tuned_logistic_validation_predictions.csv"
)

test_predictions_path = (
    model_output_directory
    / "tuned_logistic_test_predictions.csv"
)

coefficient_output_path = (
    model_output_directory
    / "tuned_logistic_coefficients.csv"
)

model_summary_path = (
    model_output_directory
    / "baseline_model_summary.csv"
)

model_bundle_path = (
    saved_model_directory
    / "tuned_logistic_baseline.joblib"
)

model_metadata_path = (
    saved_model_directory
    / "tuned_logistic_baseline_metadata.json"
)


final_baseline_comparison.to_csv(
    comparison_output_path,
    index=False,
)

final_baseline_improvement.to_csv(
    improvement_output_path,
    index=False,
)

final_validation_predictions.to_csv(
    validation_predictions_path,
    index=False,
)

final_test_predictions.to_csv(
    test_predictions_path,
    index=False,
)

tuned_logistic_coefficients.to_csv(
    coefficient_output_path,
    index=False,
)

final_model_summary.to_csv(
    model_summary_path,
    index=False,
)


baseline_model_bundle = {
    "preprocessor": preprocessor,
    "model": tuned_logistic_model,
    "feature_columns": list(feature_columns),
    "processed_feature_names": list(
        processed_feature_names
    ),
    "class_order": list(class_order),
    "best_C": float(best_logistic_C),
    "class_weight": best_logistic_class_weight,
    "class_weight_label": (
        best_logistic_class_weight_label
    ),
    "train_seasons": [
        str(season)
        for season in train_seasons
    ],
    "validation_season": str(
        validation_season
    ),
    "test_season": str(test_season),
}

joblib.dump(
    baseline_model_bundle,
    model_bundle_path,
)


baseline_model_metadata = {
    "model_name": (
        "Tuned Multinomial Logistic Regression"
    ),
    "selection_metric": "Validation Log Loss",
    "best_C": float(best_logistic_C),
    "class_weight": (
        best_logistic_class_weight
    ),
    "class_order": list(class_order),
    "model_internal_class_order": [
        str(outcome)
        for outcome
        in tuned_logistic_model.classes_
    ],
    "predictor_count": int(
        len(feature_columns)
    ),
    "training_fixtures": int(
        len(y_train)
    ),
    "validation_fixtures": int(
        len(y_validation)
    ),
    "test_fixtures": int(
        len(y_test)
    ),
    "validation_log_loss": float(
        selected_validation_row[
            "LogLoss"
        ]
    ),
    "validation_brier_score": float(
        selected_validation_row[
            "BrierScore"
        ]
    ),
    "validation_accuracy": float(
        selected_validation_row[
            "Accuracy"
        ]
    ),
    "test_log_loss": float(
        selected_test_row[
            "LogLoss"
        ]
    ),
    "test_brier_score": float(
        selected_test_row[
            "BrierScore"
        ]
    ),
    "test_accuracy": float(
        selected_test_row[
            "Accuracy"
        ]
    ),
}

with open(
    model_metadata_path,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        baseline_model_metadata,
        metadata_file,
        indent=4,
    )


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print(
    "Final baseline evaluation completed successfully."
)

print(
    "Selected model:",
    "Tuned Multinomial Logistic Regression",
)

print(
    "Selected hyperparameters:",
    f"C={best_logistic_C},",
    f"class_weight={best_logistic_class_weight_label}",
)

print(
    "Validation log loss:",
    round(
        float(
            selected_validation_row[
                "LogLoss"
            ]
        ),
        6,
    ),
)

print(
    "Test log loss:",
    round(
        float(
            selected_test_row[
                "LogLoss"
            ]
        ),
        6,
    ),
)

print(
    "\nSaved model bundle:",
    model_bundle_path,
)

print(
    "Saved model outputs:",
    model_output_directory,
)

display(final_model_summary)

display(final_baseline_comparison)

display(final_baseline_improvement)

display(largest_tuned_coefficients_by_outcome)

display(final_test_predictions.head(20))

Final baseline evaluation completed successfully.
Selected model: Tuned Multinomial Logistic Regression
Selected hyperparameters: C=0.01, class_weight=None
Validation log loss: 0.934551
Test log loss: 1.004062

Saved model bundle: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\models\tuned_logistic_baseline.joblib
Saved model outputs: C:\Users\kiera\OneDrive\Desktop\premier-league-probability-engine\data\model_outputs


,Field,Value
0,Selected model,Tuned Multinomial Logistic Regression
1,Regularisation parameter C,0.01
2,Class weighting,None
3,Training seasons,"2015-16, 2016-17, 2017-18, 2018-19, 2019-20, 2..."
4,Validation season,2023-24
5,Test season,2024-25
6,Predictor count,70
7,Training fixtures,3040
8,Validation fixtures,380
9,Test fixtures,380


,Model,Split,Fixtures,LogLoss,BrierScore,Accuracy,LogLossRank,BrierRank,AccuracyRank
0,Tuned Logistic Regression,Validation,380,0.934551,0.549322,0.592105,1,1,1
1,Multinomial Logistic Regression,Validation,380,0.941668,0.551312,0.589474,2,2,2
2,Historical Frequency,Validation,380,1.054044,0.637099,0.460526,3,3,3
3,Uniform Probability,Validation,380,1.098612,0.666667,0.460526,4,4,3
4,Tuned Logistic Regression,Test,380,1.004062,0.601040,0.505263,1,1,1
5,Multinomial Logistic Regression,Test,380,1.012463,0.604957,0.505263,2,2,1
6,Historical Frequency,Test,380,1.080909,0.655601,0.407895,3,3,2
7,Uniform Probability,Test,380,1.098612,0.666667,0.407895,4,4,2


,Split,ReferenceModel,ReferenceLogLoss,TunedLogLoss,LogLossImprovement,LogLossImprovementPercentage,ReferenceBrierScore,TunedBrierScore,BrierImprovement,BrierImprovementPercentage,ReferenceAccuracy,TunedAccuracy,AccuracyDifferencePercentagePoints,LowerLogLoss,LowerBrierScore
0,Validation,Uniform Probability,1.098612,0.934551,0.164061,14.933480,0.666667,0.549322,0.117345,17.601741,0.460526,0.592105,13.1579,True,True
1,Validation,Historical Frequency,1.054044,0.934551,0.119493,11.336624,0.637099,0.549322,0.087777,13.777608,0.460526,0.592105,13.1579,True,True
2,Validation,Multinomial Logistic Regression,0.941668,0.934551,0.007117,0.755787,0.551312,0.549322,0.001990,0.360957,0.589474,0.592105,0.2631,True,True
3,Test,Uniform Probability,1.098612,1.004062,0.094550,8.606314,0.666667,0.601040,0.065627,9.844045,0.407895,0.505263,9.7368,True,True
4,Test,Historical Frequency,1.080909,1.004062,0.076847,7.109479,0.655601,0.601040,0.054561,8.322287,0.407895,0.505263,9.7368,True,True
5,Test,Multinomial Logistic Regression,1.012463,1.004062,0.008401,0.829759,0.604957,0.601040,0.003917,0.647484,0.505263,0.505263,0.0000,True,True


,Outcome,Feature,Coefficient,AbsoluteCoefficient
0,A,EloDifference,-0.198779,0.198779
1,A,AwayEloBefore,0.142387,0.142387
2,A,HomeEloBefore,-0.141718,0.141718
3,A,HomeShortRest3,-0.055690,0.055690
4,A,PositionDifference,-0.047706,0.047706
5,A,HomeBottom3Before,-0.047126,0.047126
6,A,ShortRestDifference3,-0.046797,0.046797
7,A,AwayTop6Before,-0.046713,0.046713
8,A,AwayLeaguePosition,-0.043290,0.043290
9,A,GoalDifferenceDifference,-0.040477,0.040477


,Season,Date,HomeTeam,AwayTeam,ActualOutcome,PredictedOutcome,Probability_H,Probability_D,Probability_A,ProbabilityOfActualOutcome,MaximumProbability,CorrectPrediction,FixtureLogLoss
3420,2024-25,2024-08-16,Man United,Fulham,H,H,0.534793,0.244604,0.220603,0.534793,0.534793,True,0.625876
3421,2024-25,2024-08-17,Arsenal,Wolves,H,H,0.801993,0.119085,0.078922,0.801993,0.801993,True,0.220656
3422,2024-25,2024-08-17,Everton,Brighton,A,H,0.386258,0.275209,0.338533,0.338533,0.386258,False,1.083132
3423,2024-25,2024-08-17,Ipswich,Liverpool,A,A,0.202135,0.196755,0.601110,0.601110,0.601110,True,0.508978
3424,2024-25,2024-08-17,Newcastle,Southampton,H,H,0.610669,0.247150,0.142181,0.610669,0.610669,True,0.493201
3425,2024-25,2024-08-17,Nott'm Forest,Bournemouth,D,A,0.306124,0.305365,0.388511,0.305365,0.388511,False,1.186247
3426,2024-25,2024-08-17,West Ham,Aston Villa,A,A,0.297513,0.286474,0.416013,0.416013,0.416013,True,0.877039
3427,2024-25,2024-08-18,Brentford,Crystal Palace,H,A,0.358944,0.258447,0.382609,0.358944,0.382609,False,1.024588
3428,2024-25,2024-08-18,Chelsea,Man City,A,A,0.258615,0.179356,0.562029,0.562029,0.562029,True,0.576201
3429,2024-25,2024-08-19,Leicester,Tottenham,D,A,0.280648,0.257936,0.461416,0.257936,0.461416,False,1.355043


## 11. Notebook Summary and Conclusions

### Research Question

This notebook investigated whether engineered pre-match football features allow a simple and interpretable model to produce better Premier League outcome probabilities than naive forecasting methods.

The central question was:

> Do the engineered pre-match features allow a multinomial logistic-regression model to outperform forecasts based only on equal probabilities or historical outcome frequencies?

The results answer this question positively.

---

### Modelling Framework

The processed dataset was divided chronologically into:

- 3,040 training fixtures;
- 380 validation fixtures from the 2023–24 season;
- 380 test fixtures from the 2024–25 season.

The model used 70 numeric predictors containing information available before kickoff, including:

- Elo ratings and Elo differences;
- recent general and venue-specific form;
- pre-match league-table position;
- goal-difference and points differences;
- rest and fixture-congestion measures;
- season-stage indicators.

All preprocessing steps were fitted using the training data only. Missing values were imputed using training-set medians, and predictors were standardised before fitting the logistic-regression models.

---

### Baseline Performance

Four probability forecasts were compared:

1. uniform probabilities;
2. historical training-set outcome frequencies;
3. untuned multinomial logistic regression;
4. tuned multinomial logistic regression.

| Model | Split | Log Loss | Brier Score | Accuracy |
|---|---|---:|---:|---:|
| Uniform Probability | Validation | 1.098612 | 0.666667 | 0.460526 |
| Historical Frequency | Validation | 1.054044 | 0.637099 | 0.460526 |
| Untuned Logistic Regression | Validation | 0.941668 | 0.551312 | 0.589474 |
| **Tuned Logistic Regression** | **Validation** | **0.934551** | **0.549322** | **0.592105** |
| Uniform Probability | Test | 1.098612 | 0.666667 | 0.407895 |
| Historical Frequency | Test | 1.080909 | 0.655601 | 0.407895 |
| Untuned Logistic Regression | Test | 1.012463 | 0.604957 | 0.505263 |
| **Tuned Logistic Regression** | **Test** | **1.004062** | **0.601040** | **0.505263** |

Lower log loss and Brier score indicate better probability forecasts.

---

### Hyperparameter Selection

The hyperparameter search evaluated 12 combinations of:

$$
C \in
\{
0.001,\ 0.01,\ 0.1,\ 1,\ 10,\ 100
\}
$$

and:

$$
\text{class weight}
\in
\{
\text{None},\ \text{balanced}
\}.
$$

The selected configuration was:

$$
C=0.01
$$

with no class weighting.

Because $C$ is the inverse of regularisation strength, the selected value applies substantially stronger coefficient shrinkage than the default value of $C=1$.

This suggests that regularisation was beneficial because the feature set contains several related measurements of team strength, form and league-table performance.

Balanced class weighting was not selected. Although class weighting can increase the influence of less frequent outcomes such as draws, it did not improve validation probability quality.

---

### Improvement Over Naive Forecasts

On the validation season, the tuned model reduced log loss by approximately:

- **14.9%** relative to uniform probabilities;
- **11.3%** relative to historical outcome frequencies;
- **0.8%** relative to the untuned logistic-regression model.

On the held-out test season, the model reduced log loss by approximately:

- **8.6%** relative to uniform probabilities;
- **7.1%** relative to historical outcome frequencies;
- **0.8%** relative to the untuned logistic-regression model.

The reduction in performance between validation and test is expected because football environments change through time. The test season contains changes in team quality, promoted clubs, managers, player availability and other conditions not present in the earlier seasons.

Crucially, the model continued to outperform both naive benchmarks on the test season. This indicates that the engineered features contain predictive relationships that generalise beyond the data used for training and hyperparameter selection.

---

### Feature Interpretation

The largest coefficients in the tuned model include variables associated with:

- home and away Elo ratings;
- the difference between home and away Elo;
- league-position differences;
- points and goal-difference measures;
- venue-specific form;
- rest and fixture congestion;
- league-position categories.

The prominence of Elo-related variables is consistent with the expectation that relative team strength is one of the most important determinants of match outcomes.

Coefficient magnitude should nevertheless be interpreted cautiously. Several predictors measure related concepts and are therefore correlated. The coefficients describe relationships within the fitted linear model and should not be interpreted as isolated causal effects.

---

### Probability and Implementation Verification

The fitted model internally stores its outcome classes in lexicographic order:

$$
(A,D,H).
$$

The project displays probabilities in the more intuitive order:

$$
(H,D,A).
$$

The notebook explicitly reorders the probability columns before evaluation and verifies that:

- each probability corresponds to the correct outcome;
- probability rows remain aligned with their fixtures;
- every row sums to one;
- manual and automated log-loss calculations agree;
- the optimiser converged successfully;
- no non-finite model coefficients were produced.

These checks confirm that the final performance results are not caused by class-order or index-alignment errors.

---

### Final Model Selection

The tuned multinomial logistic-regression model is selected as the project's official linear baseline.

It provides:

- fixture-specific probability forecasts;
- meaningful improvement over naive benchmarks;
- interpretable coefficients;
- strong computational efficiency;
- a reproducible benchmark for subsequent modelling.

The final model achieved:

$$
\operatorname{LogLoss}_{\text{validation}}
=
0.934551
$$

and:

$$
\operatorname{LogLoss}_{\text{test}}
=
1.004062.
$$

Any more complex model introduced later must demonstrate a consistent improvement over this baseline to justify its additional complexity.

---

### Saved Outputs

The notebook saves:

- validation and test fixture-level probabilities;
- model-comparison and improvement tables;
- tuned logistic-regression coefficients;
- a compact model-summary table;
- the fitted preprocessing pipeline;
- the fitted tuned logistic-regression model;
- feature names, class ordering and model metadata.

These artefacts allow later notebooks to reproduce the baseline without refitting the complete model.

---

### Next Stage

The next notebook will investigate whether nonlinear models can improve upon the logistic-regression baseline.

Tree-based models can represent interactions and nonlinear relationships that logistic regression cannot capture directly. The principal benchmark for the next stage is therefore:

$$
\operatorname{LogLoss}_{\text{validation}}
<
0.934551,
$$

while maintaining robust performance on future chronological data.

The logistic-regression baseline has established that the engineered pre-match features contain genuine predictive signal. The next stage will determine whether a more flexible model can use that information more effectively.